# Dreaddit Stress Detection: Experiments

This notebook contains the original model comparisons and a matched ten-seed
follow-up across MPNet, MiniLM, and DistilBERT. Run from the beginning in
Google Colab with a CUDA GPU (an A100 was used for the reported study).
The official Dreaddit data are fetched if absent. Matched-study results are
written to `/content/reviewer_revision_outputs/`.


In [ ]:

import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from importlib import invalidate_caches

REQUIRED_PACKAGES = [
    "transformers==5.16.1",
    "sentence-transformers==6.0.1",
    "datasets==4.8.4",
    "peft==0.20.0",
    "accelerate>=1.0,<2",
    "sentencepiece",
    "plotnine",
    "gensim",
    "xgboost",
    "lime",
]

print("Checking notebook dependencies...")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        *REQUIRED_PACKAGES,
    ]
)

from packaging.version import Version
try:
    installed_torchao_version = version("torchao")
except PackageNotFoundError:
    installed_torchao_version = None

if (installed_torchao_version is not None
        and Version(installed_torchao_version) < Version("0.16.0")):
    print("Removing incompatible optional torchao "
          f"{installed_torchao_version} (PEFT requires >=0.16.0).")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall",
                           "-y", "torchao"])
    invalidate_caches()
    try:
        remaining_torchao_version = version("torchao")
    except PackageNotFoundError:
        remaining_torchao_version = None
    if remaining_torchao_version is not None:
        raise RuntimeError("Old torchao is still installed. In Colab, "
                           "restart the session and run the first cell.")
    print("TorchAO compatibility check passed (unused package removed).")
else:
    print("TorchAO compatibility check passed:",
          installed_torchao_version or "not installed")

try:
    import transformers
    import sentence_transformers
    import datasets
    import peft
    import accelerate
    import pyarrow
    import plotnine
    import gensim
    import xgboost
    import lime

    from transformers import (
        AutoTokenizer,
        AutoModel,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        TrainingArguments,
        Trainer,
        set_seed,
    )
    from sentence_transformers import SentenceTransformer
    from datasets import Dataset, DatasetDict
    from peft import LoraConfig, TaskType, get_peft_model
    from lime.lime_text import LimeTextExplainer

except Exception as exc:
    raise RuntimeError(
        "The dependency installation completed, but the import verification "
        "failed. In Colab this normally means this runtime had already loaded "
        "a different Hugging Face package version before this notebook was "
        "run. Use Runtime > Restart session, then Run all from the top. "
        f"Original import error: {type(exc).__name__}: {exc}"
    ) from exc

print("Dependency check passed.")
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("pyarrow:", pyarrow.__version__)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string

from plotnine import (
    ggplot,
    aes,
    geom_bar,
    geom_histogram,
    geom_density,
    geom_boxplot,
    geom_col,
    facet_wrap,
    position_dodge,
    labs,
    theme,
    theme_bw,
    theme_minimal,
    element_text,
    scale_fill_manual,
    scale_color_manual,
    coord_flip
)


In [ ]:

import sys
import platform
from importlib.metadata import version, PackageNotFoundError

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "xgboost",
    "gensim",
    "matplotlib",
    "plotnine",
    "lime",
    "torch",
    "transformers",
    "sentence-transformers",
    "datasets",
    "peft",
    "accelerate",
    "pyarrow",
]

print("=" * 60)
print("SOFTWARE ENVIRONMENT")
print("=" * 60)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n" + "=" * 60)
print("HARDWARE / CUDA")
print("=" * 60)

try:
    import torch

    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"PyTorch CUDA version: {torch.version.cuda}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"GPU memory: {props.total_memory / (1024**3):.2f} GB")

except Exception as e:
    print(f"Could not retrieve PyTorch/CUDA information: {e}")

In [ ]:


from importlib.metadata import version

expected_versions = {
    "transformers": "5.16.1",
    "sentence-transformers": "6.0.1",
    "datasets": "4.8.4",
    "peft": "0.20.0",
}

for package_name, expected_version in expected_versions.items():
    installed_version = version(package_name)
    if installed_version != expected_version:
        raise RuntimeError(
            f"{package_name} changed during this runtime: "
            f"expected {expected_version}, found {installed_version}. "
            "Restart the session and run the notebook from the top."
        )

print("Hugging Face dependency stack is stable.")


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from io import BytesIO
from zipfile import ZipFile
from hashlib import sha256

needed = (Path('dreaddit-train.csv'), Path('dreaddit-test.csv'))
if not all(p.exists() for p in needed):
    source_urls = [
        'https://www.cs.columbia.edu/~eturcan/data/dreaddit.zip',
        'http://www.cs.columbia.edu/~eturcan/data/dreaddit.zip',
    ]
    failures = []
    for archive_url in source_urls:
        try:
            request = Request(archive_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urlopen(request, timeout=180) as response:
                payload = response.read()
            with ZipFile(BytesIO(payload)) as archive:
                for path in needed:
                    choices = [name for name in archive.namelist()
                               if Path(name).name.lower() == path.name.lower()]
                    if not choices:
                        raise FileNotFoundError(f'{path.name} missing from published archive')
                    path.write_bytes(archive.read(choices[0]))
            print('Downloaded original Dreaddit CSVs from:', archive_url)
            break
        except Exception as error:
            failures.append(f'{archive_url}: {error}')
    if not all(path.exists() for path in needed):
        raise FileNotFoundError(
            'Could not fetch official CSVs. Upload dreaddit-train.csv and '
            'dreaddit-test.csv into /content and rerun this cell. Details: '            + ' | '.join(failures)
        )

for path in needed:
    print(path.name, 'bytes=', path.stat().st_size,
          'sha256=', sha256(path.read_bytes()).hexdigest())

In [ ]:
train_raw = pd.read_csv("dreaddit-train.csv")
test_raw = pd.read_csv("dreaddit-test.csv")

train = train_raw.copy()
test = test_raw.copy()

print("Official training-set shape:", train_raw.shape)
print("Official testing-set shape:", test_raw.shape)


In [ ]:
print("Training columns:")
print(train.columns.tolist())

print("\nTesting columns:")
print(test.columns.tolist())

print("\nColumns match:", train.columns.tolist() == test.columns.tolist())

print("\nFirst 5 training rows:")
display(train.head())

print("\nFirst 5 testing rows:")
display(test.head())

In [ ]:
print("Training labels:")
print(train["label"].value_counts().sort_index())

print("\nTesting labels:")
print(test["label"].value_counts().sort_index())

print("\nUnique training labels:")
print(train["label"].unique())

print("\nUnique testing labels:")
print(test["label"].unique())

In [ ]:
def inspect_dataset(dataframe, dataset_name):
    print(f"{dataset_name} number of posts:", len(dataframe))

    print(f"\n{dataset_name} missing values:")
    print(dataframe[["text", "label"]].isnull().sum())

    print(f"\n{dataset_name} duplicate rows:")
    print(dataframe.duplicated().sum())

    print(f"\n{dataset_name} duplicate posts:")
    print(dataframe["text"].duplicated().sum())


inspect_dataset(train_raw, "Official training set")

print("\n" + "=" * 60 + "\n")

inspect_dataset(test_raw, "Official testing set")


In [ ]:
train_texts = set(train_raw["text"].dropna())
test_texts = set(test_raw["text"].dropna())

overlapping_posts = train_texts.intersection(test_texts)

print("Posts appearing in both official train and test:", len(overlapping_posts))

duplicate_training_texts = train["text"].duplicated().sum()

train = (
    train
    .drop_duplicates(subset=["text"], keep="first")
    .reset_index(drop=True)
)

overlap_mask = train["text"].isin(test_texts)
training_test_overlap_rows = int(overlap_mask.sum())

train = train.loc[~overlap_mask].reset_index(drop=True)
test = test_raw.copy().reset_index(drop=True)

print("\nPreprocessing summary:")
print("Duplicate training rows removed by text:", duplicate_training_texts)
print("Training rows removed because text appears in official test:", training_test_overlap_rows)
print("Cleaned training-set shape:", train.shape)
print("Official test-set shape retained:", test.shape)

print("\nPost-cleaning checks:")
print("Duplicate training texts remaining:", train["text"].duplicated().sum())
print(
    "Train/test text overlap remaining:",
    len(set(train["text"].dropna()).intersection(set(test["text"].dropna())))
)

assert train["text"].duplicated().sum() == 0
assert len(set(train["text"].dropna()).intersection(set(test["text"].dropna()))) == 0


In [ ]:
official_train_class_summary = (
    train_raw["label"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

official_train_class_summary["percentage"] = (
    official_train_class_summary["count"] / len(train_raw) * 100
)

test_class_summary = (
    test_raw["label"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

test_class_summary["percentage"] = (
    test_class_summary["count"] / len(test_raw) * 100
)

cleaned_train_class_summary = (
    train["label"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="count")
)

cleaned_train_class_summary["percentage"] = (
    cleaned_train_class_summary["count"] / len(train) * 100
)

print("Official training class summary:")
display(official_train_class_summary)

print("Cleaned training class summary used for analysis/modeling:")
display(cleaned_train_class_summary)

print("Official testing class summary:")
display(test_class_summary)


In [ ]:
for dataframe in [train, test]:
    dataframe["word_count"] = dataframe["text"].fillna("").str.split().str.len()
    dataframe["char_count"] = dataframe["text"].fillna("").str.len()

print("Training-set post length:")
print(train["word_count"].describe())

print("\nTesting-set post length:")
print(test["word_count"].describe())

In [ ]:
print("Training word-count summary by label:")
display(train.groupby("label")["word_count"].describe())

print("\nTesting word-count summary by label:")
display(test.groupby("label")["word_count"].describe())

In [ ]:
class_name_mapping = {
    0: "Non-stress",
    1: "Stress/Stress-related"
}

train["class_name"] = train["label"].map(class_name_mapping)
test["class_name"] = test["label"].map(class_name_mapping)

train["dataset"] = "Training"
test["dataset"] = "Testing"

In [ ]:
class_distribution_plot_data = pd.concat(
    [
        official_train_class_summary.assign(dataset="Training"),
        test_class_summary.assign(dataset="Testing")
    ],
    ignore_index=True
)

class_distribution_plot_data["class_name"] = (
    class_distribution_plot_data["label"].map(class_name_mapping)
)

class_distribution_plot = (
    ggplot(
        class_distribution_plot_data,
        aes(
            x="class_name",
            y="percentage",
            fill="dataset"
        )
    )
    + geom_col(
        position=position_dodge(width=0.8),
        width=0.7
    )
    + labs(
        title="Class Distribution in the Official Dreaddit Split",
        x="Class",
        y="Percentage of Posts",
        fill="Dataset"
    )
    + theme_bw()
    + theme(
        figure_size=(8, 5),
        axis_text_x=element_text(rotation=0),
        plot_title=element_text(weight="bold")
    )
)

class_distribution_plot

In [ ]:
length_plot_data = pd.concat(
    [
        train[["word_count", "dataset"]],
        test[["word_count", "dataset"]]
    ],
    ignore_index=True
)

post_length_distribution_plot = (
    ggplot(
        length_plot_data,
        aes(x="word_count")
    )
    + geom_histogram(
        bins=30,
        alpha=0.8
    )
    + facet_wrap("~dataset", nrow=1)
    + labs(
        title="Distribution of Post Lengths in the Official Dreaddit Split",
        x="Post Length in Words",
        y="Number of Posts"
    )
    + theme_bw()
    + theme(
        figure_size=(11, 4.5),
        plot_title=element_text(weight="bold")
    )
)

post_length_distribution_plot

In [ ]:
plt.figure(figsize=(7,5))
train.boxplot(column="word_count", by="class_name")
plt.title("Post Length by Class")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Post Length (Word Count)")
plt.show()

In [ ]:
def count_sentences(text):
    count = 0

    for char in text:
        if char == "." or char == "!" or char == "?":
            count += 1

    return count

train["sentence_count"] = train["text"].apply(count_sentences)


def average_word_length(text):
    words = text.split()

    if len(words) == 0:
        return 0

    total_letters = 0

    for word in words:
        total_letters += len(word)

    return total_letters / len(words)

train["avg_word_length"] = train["text"].apply(average_word_length)


def count_punctuation(text):
    count = 0

    for char in text:
        if char in string.punctuation:
            count += 1

    return count

train["punctuation_count"] = train["text"].apply(count_punctuation)


def count_question_marks(text):
    return text.count("?")

train["question_marks"] = train["text"].apply(count_question_marks)


def count_exclamation_marks(text):
    return text.count("!")

train["exclamation_marks"] = train["text"].apply(count_exclamation_marks)


def capital_letter_percentage(text):
    if len(text) == 0:
        return 0

    capital_letters = 0

    for char in text:
        if char.isupper():
            capital_letters += 1

    return (capital_letters / len(text)) * 100

train["capital_percent"] = train["text"].apply(capital_letter_percentage)

In [ ]:
vocabulary_features = [
    "word_count",
    "char_count",
    "sentence_count",
    "avg_word_length",
    "punctuation_count",
    "question_marks",
    "exclamation_marks",
    "capital_percent"
]

vocab_summary = train.groupby("class_name")[vocabulary_features].mean()
display(vocab_summary)

In [ ]:
import re
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
print(train["class_name"].value_counts())

In [ ]:
basic_stopwords = {
    "the", "and", "is", "of", "to", "a", "an", "in", "it", "that", "this",
    "for", "on", "with", "as", "was", "were", "be", "been", "are", "am",
    "at", "by", "from", "or"
}

def get_words_without_stopwords(text):
    text = text.lower()

    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    cleaned_words = []

    for word in words:
        if word not in basic_stopwords:
            cleaned_words.append(word)

    return cleaned_words

In [ ]:
train["unigram_tokens"] = train["text"].apply(get_words_without_stopwords)

display(train[["text", "class_name", "unigram_tokens"]].head())

In [ ]:
def get_top_unigrams(dataframe, class_name, number_of_words):
    class_posts = dataframe[dataframe["class_name"] == class_name]

    all_words = []

    for token_list in class_posts["unigram_tokens"]:
        for word in token_list:
            all_words.append(word)

    word_counts = Counter(all_words)

    top_words = word_counts.most_common(number_of_words)

    result = pd.DataFrame(top_words, columns=["word", "count"])

    return result

In [ ]:
top_stress_unigrams = get_top_unigrams(train, "Stress/Stress-related", 20)
top_nonstress_unigrams = get_top_unigrams(train, "Non-stress", 20)

print("Top 20 Unigrams - Stress/Stress-related Posts")
display(top_stress_unigrams)

print("Top 20 Unigrams - Non-stress Posts")
display(top_nonstress_unigrams)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_stress_unigrams["word"], top_stress_unigrams["count"])
plt.title("Top 20 Unigrams in Stress/Stress-related Posts")
plt.xlabel("Word")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(top_nonstress_unigrams["word"], top_nonstress_unigrams["count"])
plt.title("Top 20 Unigrams in Non-stress Posts")
plt.xlabel("Word")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
def get_words_for_bigrams(text):
    text = text.lower()

    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

    return words

In [ ]:
def create_bigrams(word_list):
    bigrams = []

    for index in range(len(word_list) - 1):
        first_word = word_list[index]
        second_word = word_list[index + 1]

        bigram = first_word + " " + second_word

        bigrams.append(bigram)

    return bigrams

In [ ]:
train["bigram_words"] = train["text"].apply(get_words_for_bigrams)
train["bigrams"] = train["bigram_words"].apply(create_bigrams)

display(train[["text", "class_name", "bigrams"]].head())

In [ ]:
def get_top_bigrams(dataframe, class_name, number_of_bigrams):
    class_posts = dataframe[dataframe["class_name"] == class_name]

    all_bigrams = []

    for bigram_list in class_posts["bigrams"]:
        for bigram in bigram_list:
            all_bigrams.append(bigram)

    bigram_counts = Counter(all_bigrams)

    top_bigrams = bigram_counts.most_common(number_of_bigrams)

    result = pd.DataFrame(top_bigrams, columns=["bigram", "count"])

    return result

In [ ]:
top_stress_bigrams = get_top_bigrams(train, "Stress/Stress-related", 20)
top_nonstress_bigrams = get_top_bigrams(train, "Non-stress", 20)

print("Top 20 Bigrams - Stress/Stress-related Posts")
display(top_stress_bigrams)

print("Top 20 Bigrams - Non-stress Posts")
display(top_nonstress_bigrams)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_stress_bigrams["bigram"], top_stress_bigrams["count"])
plt.title("Top 20 Bigrams in Stress/Stress-related Posts")
plt.xlabel("Bigram")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(top_nonstress_bigrams["bigram"], top_nonstress_bigrams["count"])
plt.title("Top 20 Bigrams in Non-stress Posts")
plt.xlabel("Bigram")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=list(basic_stopwords),
    ngram_range=(1, 2),
    max_features=5000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(train["text"])

tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of TF-IDF terms:", len(tfidf_terms))

In [ ]:
stress_rows = (train["class_name"] == "Stress/Stress-related").values
nonstress_rows = (train["class_name"] == "Non-stress").values

stress_average_tfidf = tfidf_matrix[stress_rows].mean(axis=0)
nonstress_average_tfidf = tfidf_matrix[nonstress_rows].mean(axis=0)

stress_average_tfidf = stress_average_tfidf.tolist()[0]
nonstress_average_tfidf = nonstress_average_tfidf.tolist()[0]

In [ ]:
tfidf_scores = pd.DataFrame({
    "term": tfidf_terms,
    "stress_average_tfidf": stress_average_tfidf,
    "nonstress_average_tfidf": nonstress_average_tfidf
})

tfidf_scores["stress_distinctiveness"] = (
    tfidf_scores["stress_average_tfidf"] - tfidf_scores["nonstress_average_tfidf"]
)

tfidf_scores["nonstress_distinctiveness"] = (
    tfidf_scores["nonstress_average_tfidf"] - tfidf_scores["stress_average_tfidf"]
)

display(tfidf_scores.head())

In [ ]:
top_stress_tfidf = (
    tfidf_scores
    .sort_values("stress_distinctiveness", ascending=False)
    .head(20)
    [["term", "stress_average_tfidf", "nonstress_average_tfidf", "stress_distinctiveness"]]
)

top_nonstress_tfidf = (
    tfidf_scores
    .sort_values("nonstress_distinctiveness", ascending=False)
    .head(20)
    [["term", "nonstress_average_tfidf", "stress_average_tfidf", "nonstress_distinctiveness"]]
)

print("Top 20 Distinctive TF-IDF Terms - Stress/Stress-related Posts")
display(top_stress_tfidf)

print("Top 20 Distinctive TF-IDF Terms - Non-stress Posts")
display(top_nonstress_tfidf)

In [ ]:
tfidf_comparison_table = pd.DataFrame({
    "stress_term": top_stress_tfidf["term"].values,
    "stress_distinctiveness": top_stress_tfidf["stress_distinctiveness"].values,
    "nonstress_term": top_nonstress_tfidf["term"].values,
    "nonstress_distinctiveness": top_nonstress_tfidf["nonstress_distinctiveness"].values
})

display(tfidf_comparison_table)

In [ ]:
def count_syllables_in_word(word):
    word = word.lower()
    word = re.sub(r"[^a-z]", "", word)

    if len(word) == 0:
        return 0

    vowels = "aeiouy"
    syllable_count = 0
    previous_character_was_vowel = False

    for character in word:
        if character in vowels:
            if previous_character_was_vowel == False:
                syllable_count += 1
            previous_character_was_vowel = True
        else:
            previous_character_was_vowel = False

    if word.endswith("e") and syllable_count > 1:
        syllable_count -= 1

    if syllable_count == 0:
        syllable_count = 1

    return syllable_count

In [ ]:
def count_words_for_readability(text):
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())
    return len(words)

def count_sentences_for_readability(text):
    sentence_endings = re.findall(r"[.!?]+", text)

    if len(sentence_endings) == 0:
        return 1

    return len(sentence_endings)

def count_total_syllables(text):
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

    total_syllables = 0

    for word in words:
        total_syllables += count_syllables_in_word(word)

    return total_syllables

In [ ]:
def flesch_kincaid_grade_level(text):
    word_count = count_words_for_readability(text)
    sentence_count = count_sentences_for_readability(text)
    syllable_count = count_total_syllables(text)

    if word_count == 0:
        return 0

    grade_level = (
        0.39 * (word_count / sentence_count)
        + 11.8 * (syllable_count / word_count)
        - 15.59
    )

    return grade_level

train["fk_grade_level"] = train["text"].apply(flesch_kincaid_grade_level)

display(train[["text", "class_name", "fk_grade_level"]].head())

In [ ]:
readability_summary = train.groupby("class_name")["fk_grade_level"].mean()

print("Average Flesch-Kincaid Grade Level by Class:")
display(readability_summary)

In [ ]:
plt.figure(figsize=(7, 5))
train.boxplot(column="fk_grade_level", by="class_name")
plt.title("Flesch-Kincaid Grade Level by Class")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Flesch-Kincaid Grade Level")
plt.show()

In [ ]:

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from xgboost import XGBClassifier


In [ ]:
model_source_df = (
    train[["text", "label"]]
    .dropna(subset=["text", "label"])
    .reset_index(drop=True)
)

model_test_df = (
    test[["text", "label"]]
    .dropna(subset=["text", "label"])
    .reset_index(drop=True)
)

model_source_df["text"] = model_source_df["text"].astype(str)
model_test_df["text"] = model_test_df["text"].astype(str)

model_source_df["label"] = model_source_df["label"].astype(int)
model_test_df["label"] = model_test_df["label"].astype(int)

model_train_df, model_val_df = train_test_split(
    model_source_df,
    test_size=0.15,
    stratify=model_source_df["label"],
    random_state=42
)

model_train_df = model_train_df.reset_index(drop=True)
model_val_df = model_val_df.reset_index(drop=True)

print("Cleaned modeling pool:", model_source_df.shape)
print("Canonical training split:", model_train_df.shape)
print("Canonical validation split:", model_val_df.shape)
print("Official test split:", model_test_df.shape)

print("\nCanonical training label balance:")
print(model_train_df["label"].value_counts(normalize=True) * 100)

print("\nCanonical validation label balance:")
print(model_val_df["label"].value_counts(normalize=True) * 100)

print("\nOfficial test label balance:")
print(model_test_df["label"].value_counts(normalize=True) * 100)


In [ ]:
def tokenize_text(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    return tokens

model_train_df["tokens"] = model_train_df["text"].apply(tokenize_text)
model_val_df["tokens"] = model_val_df["text"].apply(tokenize_text)
model_test_df["tokens"] = model_test_df["text"].apply(tokenize_text)

display(model_train_df[["text", "label", "tokens"]].head())


In [ ]:
X_tokens_train = model_train_df["tokens"]
X_tokens_val = model_val_df["tokens"]
X_tokens_test = model_test_df["tokens"]

y_train = model_train_df["label"]
y_val = model_val_df["label"]
y_test = model_test_df["label"]

print("Training size:", len(X_tokens_train))
print("Validation size:", len(X_tokens_val))
print("Official testing size:", len(X_tokens_test))

print("\nTraining label balance:")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation label balance:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTesting label balance:")
print(y_test.value_counts(normalize=True) * 100)


In [ ]:
vector_size = 100

w2v_model = Word2Vec(
    sentences=X_tokens_train.tolist(),
    vector_size=vector_size,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

print("Vocabulary size:", len(w2v_model.wv))

In [ ]:
def average_word2vec(tokens, model, vector_size):
    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)

X_train_w2v = np.vstack([
    average_word2vec(tokens, w2v_model, vector_size)
    for tokens in X_tokens_train
])

X_val_w2v = np.vstack([
    average_word2vec(tokens, w2v_model, vector_size)
    for tokens in X_tokens_val
])

X_test_w2v = np.vstack([
    average_word2vec(tokens, w2v_model, vector_size)
    for tokens in X_tokens_test
])

print("Word2Vec training matrix shape:", X_train_w2v.shape)
print("Word2Vec validation matrix shape:", X_val_w2v.shape)
print("Word2Vec official testing matrix shape:", X_test_w2v.shape)


In [ ]:
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_score)
    else:
        auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    results = {
        "model_name": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "auc": auc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return results

In [ ]:
baseline_models = {}

rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train_w2v, y_train)
baseline_models["Random Forest baseline"] = rf_baseline


xgb_baseline = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    scale_pos_weight=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_baseline.fit(X_train_w2v, y_train)
baseline_models["XGBoost baseline"] = xgb_baseline


baseline_results = []

for name, model in baseline_models.items():
    baseline_results.append(evaluate_model(name, model, X_test_w2v, y_test))

baseline_results_df = pd.DataFrame(baseline_results)
display(baseline_results_df)

In [ ]:
for name, model in baseline_models.items():
    print("=" * 80)
    print(name)
    print("=" * 80)

    y_pred = model.predict(X_test_w2v)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["Non-stress", "Stress/Stress-related"],
        digits=3
    ))

    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test_w2v,
        y_test,
        display_labels=["Non-stress", "Stress"],
        cmap="Blues"
    )
    plt.title(f"Confusion Matrix: {name}")
    plt.show()

    RocCurveDisplay.from_estimator(
        model,
        X_test_w2v,
        y_test
    )
    plt.title(f"ROC Curve: {name}")
    plt.show()

In [ ]:
from itertools import product

all_results = []
trained_models = {}


rf_n_estimators_options = [100, 200]
rf_max_depth_options = [None, 10, 20]
rf_class_weight_options = [None, "balanced"]

for n_estimators, max_depth, class_weight in product(
    rf_n_estimators_options,
    rf_max_depth_options,
    rf_class_weight_options
):
    model_name = (
        f"RF | n_estimators={n_estimators} | "
        f"max_depth={max_depth} | class_weight={class_weight}"
    )

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_w2v, y_train)
    trained_models[model_name] = model

    result = evaluate_model(model_name, model, X_val_w2v, y_val)
    result["model_type"] = "Random Forest"
    result["n_estimators"] = n_estimators
    result["max_depth"] = max_depth
    result["class_weight"] = class_weight
    result["learning_rate"] = None
    result["scale_pos_weight"] = None

    all_results.append(result)



xgb_n_estimators_options = [100, 200]
xgb_max_depth_options = [3, 5]
xgb_learning_rate_options = [0.05, 0.1]
xgb_scale_pos_weight_options = [1, 0.6]

for n_estimators, max_depth, learning_rate, scale_pos_weight in product(
    xgb_n_estimators_options,
    xgb_max_depth_options,
    xgb_learning_rate_options,
    xgb_scale_pos_weight_options
):
    model_name = (
        f"XGB | n_estimators={n_estimators} | "
        f"max_depth={max_depth} | learning_rate={learning_rate} | "
        f"scale_pos_weight={scale_pos_weight}"
    )

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_w2v, y_train)
    trained_models[model_name] = model

    result = evaluate_model(model_name, model, X_val_w2v, y_val)
    result["model_type"] = "XGBoost"
    result["n_estimators"] = n_estimators
    result["max_depth"] = max_depth
    result["class_weight"] = None
    result["learning_rate"] = learning_rate
    result["scale_pos_weight"] = scale_pos_weight

    all_results.append(result)


results_df = pd.DataFrame(all_results)

results_df_sorted = results_df.sort_values(
    by=["f1", "recall", "auc"],
    ascending=False
).reset_index(drop=True)

print("Hyperparameter results on the validation split:")
display(results_df_sorted)


In [ ]:
best_model_name = results_df_sorted.loc[0, "model_name"]
best_model = trained_models[best_model_name]

print("Best model selected by validation F1-score:")
print(best_model_name)

y_pred_best = best_model.predict(X_test_w2v)

print("\nOfficial test-set classification report:")
print(classification_report(
    y_test,
    y_pred_best,
    target_names=["Non-stress", "Stress/Stress-related"],
    digits=3
))

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test_w2v,
    y_test,
    display_labels=["Non-stress", "Stress"],
    cmap="Blues"
)
plt.title(f"Best Model Confusion Matrix: {best_model_name}")
plt.show()

RocCurveDisplay.from_estimator(
    best_model,
    X_test_w2v,
    y_test
)
plt.title(f"Best Model ROC Curve: {best_model_name}")
plt.show()

best_word2vec_test_result = evaluate_model(
    best_model_name,
    best_model,
    X_test_w2v,
    y_test
)

display(pd.DataFrame([best_word2vec_test_result]))


## Pretrained Transformer Embeddings


In [ ]:

print("sentence-transformers / transformers / accelerate are ready.")


In [ ]:


import pyarrow
import datasets
import transformers
import sentence_transformers

print("pyarrow:", pyarrow.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("Dependency stack remains intact.")


In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

## Pretrained Transformer Embeddings

Fixed MiniLM, MPNet, and mean-pooled DistilBERT embeddings are evaluated with
Random Forest and XGBoost classifiers.


In [ ]:

print("Embedding-model dependencies are ready.")


In [ ]:
import gc
import random
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

In [ ]:
embedding_train_df = model_train_df[["text", "label"]].copy()
embedding_test_df = model_test_df[["text", "label"]].copy()

X_train_text = embedding_train_df["text"].tolist()
X_test_text = embedding_test_df["text"].tolist()

y_train_embeddings = embedding_train_df["label"].to_numpy()
y_test_embeddings = embedding_test_df["label"].to_numpy()

print("Training posts:", len(X_train_text))
print("Official testing posts:", len(X_test_text))


In [ ]:
embedding_results = []
embedding_models = {}
embedding_test_features = {}


def evaluate_model(
    representation_name,
    classifier_name,
    model,
    X_test_features
):
    predictions = model.predict(X_test_features)
    probabilities = model.predict_proba(X_test_features)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test_embeddings,
        predictions
    ).ravel()

    result = {
        "representation": representation_name,
        "classifier": classifier_name,
        "accuracy": accuracy_score(
            y_test_embeddings,
            predictions
        ),
        "precision": precision_score(
            y_test_embeddings,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test_embeddings,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_test_embeddings,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test_embeddings,
            probabilities
        ),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    print("\n" + "=" * 70)
    print(representation_name, "+", classifier_name)
    print("=" * 70)

    print(
        classification_report(
            y_test_embeddings,
            predictions,
            target_names=["Non-stress", "Stress"],
            digits=3,
            zero_division=0
        )
    )

    return result

In [ ]:
def run_classifiers(
    representation_name,
    X_train_features,
    X_test_features
):
    random_forest = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )

    xgboost = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    )

    classifiers = {
        "Random Forest": random_forest,
        "XGBoost": xgboost
    }

    embedding_test_features[
        representation_name
    ] = X_test_features

    for classifier_name, classifier in classifiers.items():
        print(
            f"Training {representation_name} + "
            f"{classifier_name}"
        )

        classifier.fit(
            X_train_features,
            y_train_embeddings
        )

        result = evaluate_model(
            representation_name,
            classifier_name,
            classifier,
            X_test_features
        )

        embedding_results.append(result)

        experiment_name = (
            representation_name + " + " + classifier_name
        )

        embedding_models[experiment_name] = classifier

In [ ]:
minilm_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=str(device)
)

X_train_minilm = minilm_model.encode(
    X_train_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_minilm = minilm_model.encode(
    X_test_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Train shape:", X_train_minilm.shape)
print("Test shape:", X_test_minilm.shape)

In [ ]:
run_classifiers(
    "MiniLM",
    X_train_minilm,
    X_test_minilm
)

In [ ]:
del minilm_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
mpnet_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=str(device)
)

X_train_mpnet = mpnet_model.encode(
    X_train_text,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_mpnet = mpnet_model.encode(
    X_test_text,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Train shape:", X_train_mpnet.shape)
print("Test shape:", X_test_mpnet.shape)

In [ ]:
run_classifiers(
    "MPNet",
    X_train_mpnet,
    X_test_mpnet
)

In [ ]:
del mpnet_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
def create_distilbert_embeddings(
    texts,
    batch_size=16,
    max_length=256
):
    checkpoint = "distilbert-base-uncased"

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModel.from_pretrained(checkpoint)

    model.to(device)
    model.eval()

    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            outputs = model(**encoded)

        token_embeddings = outputs.last_hidden_state

        attention_mask = encoded[
            "attention_mask"
        ].unsqueeze(-1)

        expanded_mask = attention_mask.expand(
            token_embeddings.size()
        ).float()

        summed_embeddings = torch.sum(
            token_embeddings * expanded_mask,
            dim=1
        )

        token_counts = torch.clamp(
            expanded_mask.sum(dim=1),
            min=1e-9
        )

        mean_embeddings = (
            summed_embeddings / token_counts
        )

        all_embeddings.append(
            mean_embeddings.cpu().numpy()
        )

        print(
            f"Processed "
            f"{min(start + batch_size, len(texts))}"
            f"/{len(texts)}",
            end="\r"
        )

    embeddings = np.vstack(all_embeddings)

    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return embeddings

In [ ]:
X_train_distilbert = create_distilbert_embeddings(
    X_train_text,
    batch_size=16,
    max_length=256
)

X_test_distilbert = create_distilbert_embeddings(
    X_test_text,
    batch_size=16,
    max_length=256
)

print("\nTrain shape:", X_train_distilbert.shape)
print("Test shape:", X_test_distilbert.shape)

In [ ]:
run_classifiers(
    "DistilBERT",
    X_train_distilbert,
    X_test_distilbert
)

In [ ]:
embedding_results_df = pd.DataFrame(
    embedding_results
)

embedding_results_df = embedding_results_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

display(embedding_results_df)

In [ ]:
best_result = embedding_results_df.iloc[0]

best_experiment_name = (
    best_result["representation"]
    + " + "
    + best_result["classifier"]
)

print("Best experiment:")
print(best_experiment_name)

print("\nBest metrics:")
display(embedding_results_df.iloc[[0]])

In [ ]:
embedding_results_df.to_csv(
    "pretrained_transformer_embedding_results.csv",
    index=False
)

np.save("X_train_minilm.npy", X_train_minilm)
np.save("X_test_minilm.npy", X_test_minilm)

np.save("X_train_mpnet.npy", X_train_mpnet)
np.save("X_test_mpnet.npy", X_test_mpnet)

np.save(
    "X_train_distilbert.npy",
    X_train_distilbert
)

np.save(
    "X_test_distilbert.npy",
    X_test_distilbert
)

print("Results and embeddings saved.")

In [ ]:
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

for _, row in embedding_results_df.iterrows():

    representation_name = row["representation"]
    classifier_name = row["classifier"]

    experiment_name = (
        representation_name
        + " + "
        + classifier_name
    )

    model = embedding_models[experiment_name]

    X_test_features = embedding_test_features[
        representation_name
    ]

    y_pred = model.predict(X_test_features)

    print("=" * 80)
    print(experiment_name)
    print("=" * 80)

    print("\nClassification report:\n")

    print(
        classification_report(
            y_test_embeddings,
            y_pred,
            target_names=[
                "Non-stress",
                "Stress/Stress-related"
            ],
            digits=3,
            zero_division=0
        )
    )

    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test_features,
        y_test_embeddings,
        display_labels=[
            "Non-stress",
            "Stress"
        ],
        cmap="Blues"
    )

    plt.title(
        f"Confusion Matrix: {experiment_name}"
    )

    plt.show()

    RocCurveDisplay.from_estimator(
        model,
        X_test_features,
        y_test_embeddings
    )

    plt.title(
        f"ROC Curve: {experiment_name}"
    )

    plt.show()

# Transformer Fine-Tuning Experiments

This section compares two fine-tuning approaches for three pretrained transformer models:

1. Classifier-head-only training, where the pretrained transformer backbone is frozen.
2. Full fine-tuning, where all model parameters are updated.

Models:
- sentence-transformers/all-MiniLM-L6-v2
- sentence-transformers/all-mpnet-base-v2
- distilbert-base-uncased

In [ ]:


import datasets
import peft
import transformers
import accelerate
import pyarrow

print(
    "Transformer/LoRA stack ready:",
    f"transformers={transformers.__version__},",
    f"datasets={datasets.__version__},",
    f"peft={peft.__version__},",
    f"accelerate={accelerate.__version__},",
    f"pyarrow={pyarrow.__version__}",
)


In [ ]:
import os
import gc
import shutil
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from peft import LoraConfig, TaskType, get_peft_model

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("Device:", device_name)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:

transformer_source_df = model_source_df[["text", "label"]].copy()
transformer_train_df = model_train_df[["text", "label"]].copy()
transformer_val_df = model_val_df[["text", "label"]].copy()
transformer_test_df = model_test_df[["text", "label"]].copy()

print("Transformer training examples:", len(transformer_train_df))
print("Transformer validation examples:", len(transformer_val_df))
print("Official testing examples:", len(transformer_test_df))

print("\nTraining label distribution:")
print(transformer_train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(transformer_val_df["label"].value_counts().sort_index())

print("\nTesting label distribution:")
print(transformer_test_df["label"].value_counts().sort_index())


In [ ]:
MODEL_CHECKPOINTS = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "MPNet": "sentence-transformers/all-mpnet-base-v2",
    "DistilBERT": "distilbert-base-uncased"
}

METHOD_CONFIGS = {
    "classifier_head_only": {
        "learning_rate": 2e-4,
        "train_batch_size": 16,
        "gradient_accumulation_steps": 1
    },
    "full_fine_tuning": {
        "learning_rate": 2e-5,
        "train_batch_size": 8,
        "gradient_accumulation_steps": 2
    }
}

MAX_LENGTH = 256
NUMBER_OF_EPOCHS = 3

KEEP_CHECKPOINTS = True

transformer_results = []
transformer_predictions = {}

tokenizer_cache = {}
tokenized_dataset_cache = {}

print("Models:")
for model_name, checkpoint in MODEL_CHECKPOINTS.items():
    print(f"{model_name}: {checkpoint}")

print("\nTraining methods:")
for method_name, settings in METHOD_CONFIGS.items():
    print(method_name, settings)

In [ ]:
def softmax_numpy(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    exponentials = np.exp(shifted_logits)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)


def calculate_binary_metrics(labels, predictions, probabilities):
    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0
        ),
        "f1": f1_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(labels, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def compute_trainer_metrics(evaluation_prediction):
    logits, labels = evaluation_prediction

    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=1)
    probabilities = softmax_numpy(logits)[:, 1]

    return calculate_binary_metrics(
        labels,
        predictions,
        probabilities
    )

In [ ]:
def dataframe_to_huggingface_dataset(dataframe):
    formatted_dataframe = dataframe.rename(
        columns={"label": "labels"}
    )

    return Dataset.from_pandas(
        formatted_dataframe,
        preserve_index=False
    )


def get_tokenized_datasets(checkpoint):
    if checkpoint in tokenized_dataset_cache:
        return (
            tokenizer_cache[checkpoint],
            tokenized_dataset_cache[checkpoint]
        )

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint,
        use_fast=True
    )

    raw_datasets = DatasetDict({
        "train": dataframe_to_huggingface_dataset(
            transformer_train_df
        ),
        "validation": dataframe_to_huggingface_dataset(
            transformer_val_df
        ),
        "test": dataframe_to_huggingface_dataset(
            transformer_test_df
        )
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )

    tokenized_datasets = raw_datasets.map(
        tokenize_batch,
        batched=True,
        remove_columns=["text"],
        desc=f"Tokenizing with {checkpoint}"
    )

    tokenizer_cache[checkpoint] = tokenizer
    tokenized_dataset_cache[checkpoint] = tokenized_datasets

    return tokenizer, tokenized_datasets

In [ ]:
def run_transformer_experiment(
    model_short_name,
    checkpoint,
    training_method
):
    if training_method not in METHOD_CONFIGS:
        raise ValueError(
            f"Unknown training method: {training_method}"
        )

    experiment_name = f"{model_short_name}_{training_method}"
    output_directory = f"/content/{experiment_name}"

    shutil.rmtree(output_directory, ignore_errors=True)

    method_settings = METHOD_CONFIGS[training_method]

    print("=" * 90)
    print("Experiment:", experiment_name)
    print("Checkpoint:", checkpoint)
    print("Training method:", training_method)
    print("=" * 90)

    tokenizer, tokenized_datasets = get_tokenized_datasets(
        checkpoint
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=2,
        id2label={
            0: "Non-stress",
            1: "Stress/Stress-related"
        },
        label2id={
            "Non-stress": 0,
            "Stress/Stress-related": 1
        }
    )

    if training_method == "classifier_head_only":
        for parameter in model.base_model.parameters():
            parameter.requires_grad = False

    elif training_method == "full_fine_tuning":
        for parameter in model.parameters():
            parameter.requires_grad = True

    trainable_parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    total_parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    print(
        f"Trainable parameters: "
        f"{trainable_parameter_count:,} / "
        f"{total_parameter_count:,}"
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None
    )

    training_arguments = TrainingArguments(
        output_dir=output_directory,

        learning_rate=method_settings["learning_rate"],
        per_device_train_batch_size=method_settings[
            "train_batch_size"
        ],
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=method_settings[
            "gradient_accumulation_steps"
        ],

        num_train_epochs=NUMBER_OF_EPOCHS,
        weight_decay=0.01,
        warmup_steps=int(np.ceil(0.10 * NUMBER_OF_EPOCHS * np.ceil(
            len(tokenized_datasets["train"]) / (
                method_settings["train_batch_size"] *
                method_settings["gradient_accumulation_steps"]
            )
        ))),

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,

        fp16=torch.cuda.is_available(),
        optim="adamw_torch",

        seed=SEED,
        data_seed=SEED,

        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_trainer_metrics
    )

    trainer.train()

    validation_output = trainer.predict(
        tokenized_datasets["validation"]
    )

    validation_logits = validation_output.predictions
    if isinstance(validation_logits, tuple):
        validation_logits = validation_logits[0]

    validation_labels = validation_output.label_ids
    validation_predictions = np.argmax(
        validation_logits,
        axis=1
    )
    validation_probabilities = softmax_numpy(
        validation_logits
    )[:, 1]

    test_output = trainer.predict(
        tokenized_datasets["test"]
    )

    test_logits = test_output.predictions

    if isinstance(test_logits, tuple):
        test_logits = test_logits[0]

    test_labels = test_output.label_ids
    test_predictions = np.argmax(test_logits, axis=1)
    test_probabilities = softmax_numpy(test_logits)[:, 1]

    test_metrics = calculate_binary_metrics(
        test_labels,
        test_predictions,
        test_probabilities
    )

    best_validation_f1 = trainer.state.best_metric

    experiment_result = {
        "model": model_short_name,
        "checkpoint": checkpoint,
        "training_method": training_method,
        "learning_rate": method_settings["learning_rate"],
        "train_batch_size": method_settings[
            "train_batch_size"
        ],
        "gradient_accumulation_steps": method_settings[
            "gradient_accumulation_steps"
        ],
        "epochs": NUMBER_OF_EPOCHS,
        "max_length": MAX_LENGTH,
        "trainable_parameters": trainable_parameter_count,
        "total_parameters": total_parameter_count,
        "best_validation_f1": (
            float(best_validation_f1)
            if best_validation_f1 is not None
            else np.nan
        ),
        **test_metrics
    }

    transformer_results[:] = [
        existing_result
        for existing_result in transformer_results
        if not (
            existing_result["model"] == model_short_name
            and existing_result["training_method"] == training_method
        )
    ]

    transformer_results.append(experiment_result)

    transformer_predictions[experiment_name] = {
        "validation_labels": validation_labels,
        "validation_predictions": validation_predictions,
        "validation_probabilities": validation_probabilities,
        "validation_logits": validation_logits,
        "labels": test_labels,
        "predictions": test_predictions,
        "probabilities": test_probabilities,
        "test_logits": test_logits
    }

    print("\nOfficial test-set results:")
    display(pd.DataFrame([experiment_result]))

    print("\nClassification report:")
    print(
        classification_report(
            test_labels,
            test_predictions,
            labels=[0, 1],
            target_names=[
                "Non-stress",
                "Stress/Stress-related"
            ],
            digits=3,
            zero_division=0
        )
    )

    test_confusion_matrix = confusion_matrix(
        test_labels,
        test_predictions,
        labels=[0, 1]
    )

    confusion_display = ConfusionMatrixDisplay(
        confusion_matrix=test_confusion_matrix,
        display_labels=["Non-stress", "Stress"]
    )

    confusion_display.plot(cmap="Blues")
    plt.title(f"Confusion Matrix: {experiment_name}")
    plt.show()

    RocCurveDisplay.from_predictions(
        test_labels,
        test_probabilities,
        name=experiment_name
    )

    plt.title(f"ROC Curve: {experiment_name}")
    plt.show()

    if not KEEP_CHECKPOINTS:
        shutil.rmtree(output_directory, ignore_errors=True)

    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return experiment_result

In [ ]:
minilm_head_result = run_transformer_experiment(
    model_short_name="MiniLM",
    checkpoint=MODEL_CHECKPOINTS["MiniLM"],
    training_method="classifier_head_only"
)

In [ ]:
minilm_full_result = run_transformer_experiment(
    model_short_name="MiniLM",
    checkpoint=MODEL_CHECKPOINTS["MiniLM"],
    training_method="full_fine_tuning"
)

In [ ]:
minilm_results_df = pd.DataFrame([
    minilm_head_result,
    minilm_full_result
])

minilm_results_df.to_csv(
    "/content/minilm_results.csv",
    index=False
)

display(minilm_results_df)
print("Saved to /content/minilm_results.csv")

In [ ]:
mpnet_head_result = run_transformer_experiment(
    model_short_name="MPNet",
    checkpoint=MODEL_CHECKPOINTS["MPNet"],
    training_method="classifier_head_only"
)

In [ ]:
mpnet_full_result = run_transformer_experiment(
    model_short_name="MPNet",
    checkpoint=MODEL_CHECKPOINTS["MPNet"],
    training_method="full_fine_tuning"
)

In [ ]:
mpnet_results_df = pd.DataFrame([
    mpnet_head_result,
    mpnet_full_result
])

mpnet_results_df.to_csv(
    "/content/mpnet_results.csv",
    index=False
)

display(mpnet_results_df)
print("Saved to /content/mpnet_results.csv")

In [ ]:
distilbert_head_result = run_transformer_experiment(
    model_short_name="DistilBERT",
    checkpoint=MODEL_CHECKPOINTS["DistilBERT"],
    training_method="classifier_head_only"
)

In [ ]:
distilbert_full_result = run_transformer_experiment(
    model_short_name="DistilBERT",
    checkpoint=MODEL_CHECKPOINTS["DistilBERT"],
    training_method="full_fine_tuning"
)

In [ ]:
distilbert_results_df = pd.DataFrame([
    distilbert_head_result,
    distilbert_full_result
])

distilbert_results_df.to_csv(
    "/content/distilbert_results.csv",
    index=False
)

display(distilbert_results_df)
print("Saved to /content/distilbert_results.csv")

## Transformer Error Analysis

This analysis examines the false-positive/false-negative imbalance and the
textual features associated with classification errors.


## MPNet Prediction Error Analysis

Predictions are grouped by outcome, compared by textual features and n-grams,
and interpreted locally using LIME.


### Prediction Outcome Groups

Each test sample is assigned to TP, TN, FP, or FN using its reference label
and predicted class.


In [ ]:
MPNET_EXPERIMENT_NAME = "MPNet_full_fine_tuning"

if MPNET_EXPERIMENT_NAME not in transformer_predictions:
    raise RuntimeError(
        "MPNet predictions were not found. Run the MPNet full-fine-tuning "
        "experiment before running this cell."
    )

mpnet_output = transformer_predictions[MPNET_EXPERIMENT_NAME]

true_labels = np.asarray(mpnet_output["labels"]).astype(int)
predicted_labels = np.asarray(mpnet_output["predictions"]).astype(int)
stress_probabilities = np.asarray(
    mpnet_output["probabilities"]
).astype(float)

if len(transformer_test_df) != len(true_labels):
    raise ValueError(
        f"The test dataframe has {len(transformer_test_df)} rows, "
        f"but MPNet produced {len(true_labels)} predictions."
    )

if not np.array_equal(
    transformer_test_df["label"].to_numpy(),
    true_labels
):
    raise ValueError(
        "The saved prediction labels do not match the current test-data order."
    )

mpnet_error_df = test.reset_index(drop=True).copy()

mpnet_error_df["true_label"] = true_labels
mpnet_error_df["predicted_label"] = predicted_labels

mpnet_error_df["probability_stress"] = stress_probabilities
mpnet_error_df["probability_nonstress"] = 1 - stress_probabilities

mpnet_error_df["prediction_confidence"] = np.where(
    predicted_labels == 1,
    stress_probabilities,
    1 - stress_probabilities
)

def assign_prediction_group(row):
    true_label = row["true_label"]
    predicted_label = row["predicted_label"]

    if true_label == 1 and predicted_label == 1:
        return "TP"
    if true_label == 0 and predicted_label == 0:
        return "TN"
    if true_label == 0 and predicted_label == 1:
        return "FP"
    if true_label == 1 and predicted_label == 0:
        return "FN"

    raise ValueError("Labels must be either 0 or 1.")

mpnet_error_df["prediction_group"] = mpnet_error_df.apply(
    assign_prediction_group,
    axis=1
)

label_names = {
    0: "Non-stress",
    1: "Stress"
}

mpnet_error_df["true_label_name"] = (
    mpnet_error_df["true_label"].map(label_names)
)

mpnet_error_df["predicted_label_name"] = (
    mpnet_error_df["predicted_label"].map(label_names)
)

display_columns = [
    "id",
    "subreddit",
    "text",
    "true_label",
    "predicted_label",
    "probability_nonstress",
    "probability_stress",
    "prediction_confidence",
    "prediction_group"
]

display(mpnet_error_df[display_columns].head())

In [ ]:
group_order = ["TP", "TN", "FP", "FN"]

prediction_group_counts = (
    mpnet_error_df["prediction_group"]
    .value_counts()
    .reindex(group_order)
    .fillna(0)
    .astype(int)
)

print("Prediction-group counts:")
display(
    prediction_group_counts
    .rename("count")
    .reset_index()
    .rename(columns={"index": "prediction_group"})
)

tn, fp, fn, tp = confusion_matrix(
    mpnet_error_df["true_label"],
    mpnet_error_df["predicted_label"],
    labels=[0, 1]
).ravel()

expected_counts = {
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn)
}

actual_counts = prediction_group_counts.to_dict()

assert actual_counts == expected_counts, (
    f"Group counts do not match the confusion matrix.\n"
    f"Group counts: {actual_counts}\n"
    f"Confusion matrix: {expected_counts}"
)

assert prediction_group_counts.sum() == len(mpnet_error_df)
assert mpnet_error_df["prediction_group"].isna().sum() == 0

print("All checks passed.")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"FP-to-FN ratio: {fp / fn:.2f}" if fn > 0 else "FP-to-FN ratio: undefined")

mpnet_error_df.to_csv(
    "/content/mpnet_prediction_groups.csv",
    index=False
)

print("\nSaved to /content/mpnet_prediction_groups.csv")

### Text and Prediction Features

Compare length, sentence count, Flesch-Kincaid grade level, sentiment, and
prediction confidence across outcome groups.


In [ ]:
import re

def count_words(text):
    """Count word-like tokens, including contractions."""
    return len(
        re.findall(
            r"\b[\w]+(?:['’][\w]+)?\b",
            str(text)
        )
    )

def count_sentences(text):
    """
    Estimate sentence count from sentence-ending punctuation.

    A minimum of one sentence is assigned to every nonempty post.
    """
    cleaned_text = str(text).strip()

    if not cleaned_text:
        return 0

    sentence_parts = re.split(r"[.!?]+(?:\s+|$)", cleaned_text)
    nonempty_parts = [
        part for part in sentence_parts if part.strip()
    ]

    return max(1, len(nonempty_parts))

mpnet_error_df["character_count"] = (
    mpnet_error_df["text"].astype(str).str.len()
)

mpnet_error_df["word_count"] = (
    mpnet_error_df["text"].apply(count_words)
)

mpnet_error_df["sentence_count"] = (
    mpnet_error_df["text"].apply(count_sentences)
)

mpnet_error_df["flesch_kincaid_grade"] = pd.to_numeric(
    mpnet_error_df["syntax_fk_grade"],
    errors="coerce"
)

mpnet_error_df["sentiment_score"] = pd.to_numeric(
    mpnet_error_df["sentiment"],
    errors="coerce"
)

analysis_features = [
    "character_count",
    "word_count",
    "sentence_count",
    "flesch_kincaid_grade",
    "prediction_confidence",
    "sentiment_score"
]

display(
    mpnet_error_df[
        ["text", "character_count", "word_count", "sentence_count"]
    ].head()
)

print("Missing values in analysis features:")
display(mpnet_error_df[analysis_features].isna().sum())

In [ ]:

task2_means = (
    mpnet_error_df
    .groupby("prediction_group")[analysis_features]
    .mean()
    .reindex(group_order)
)

task2_medians = (
    mpnet_error_df
    .groupby("prediction_group")[analysis_features]
    .median()
    .reindex(group_order)
)

task2_counts = (
    mpnet_error_df
    .groupby("prediction_group")
    .size()
    .reindex(group_order)
    .rename("number_of_examples")
)

task2_summary = task2_means.copy()
task2_summary.insert(
    0,
    "number_of_examples",
    task2_counts
)

print("Mean values by prediction group:")
display(task2_summary.round(3))

print("\nMedian values by prediction group:")
display(task2_medians.round(3))

task2_summary.to_csv(
    "/content/mpnet_task2_group_means.csv"
)

task2_medians.to_csv(
    "/content/mpnet_task2_group_medians.csv"
)

print("Saved Task 2 tables.")

In [ ]:

task2_focused_comparisons = pd.DataFrame({
    "FP mean": task2_means.loc["FP"],
    "TN mean": task2_means.loc["TN"],
    "FP minus TN": (
        task2_means.loc["FP"] - task2_means.loc["TN"]
    ),
    "FN mean": task2_means.loc["FN"],
    "TP mean": task2_means.loc["TP"],
    "FN minus TP": (
        task2_means.loc["FN"] - task2_means.loc["TP"]
    )
})

display(task2_focused_comparisons.round(3))

task2_focused_comparisons.to_csv(
    "/content/mpnet_task2_focused_comparisons.csv"
)

In [ ]:

import matplotlib.pyplot as plt

for feature in analysis_features:
    plot_data = [
        mpnet_error_df.loc[
            mpnet_error_df["prediction_group"] == group,
            feature
        ].dropna()
        for group in group_order
    ]

    plt.figure(figsize=(8, 5))
    plt.boxplot(
        plot_data,
        tick_labels=group_order,
        showfliers=False
    )
    plt.title(
        f"{feature.replace('_', ' ').title()} by Prediction Group"
    )
    plt.xlabel("Prediction group")
    plt.ylabel(feature.replace("_", " ").title())
    plt.grid(axis="y", alpha=0.25)
    plt.show()

### Unigram and Bigram Analysis

Compare n-gram frequencies across TP, TN, FP, and FN, with rates per 1,000
to account for differing group sizes.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_group_ngrams(
    dataframe,
    group,
    ngram_range,
    top_n=20,
    stop_words="english"
):
    """
    Return the most common n-grams for one prediction group.
    """
    texts = (
        dataframe.loc[
            dataframe["prediction_group"] == group,
            "text"
        ]
        .fillna("")
        .astype(str)
        .tolist()
    )

    vectorizer = CountVectorizer(
        lowercase=True,
        stop_words=stop_words,
        ngram_range=ngram_range,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'’]+\b",
        min_df=1
    )

    document_term_matrix = vectorizer.fit_transform(texts)

    counts = np.asarray(
        document_term_matrix.sum(axis=0)
    ).ravel()

    terms = vectorizer.get_feature_names_out()
    total_ngrams = counts.sum()

    ngram_table = pd.DataFrame({
        "ngram": terms,
        "count": counts
    })

    ngram_table["per_1000_ngrams"] = (
        ngram_table["count"] / total_ngrams * 1000
        if total_ngrams > 0
        else 0
    )

    ngram_table["prediction_group"] = group

    return (
        ngram_table
        .sort_values(
            ["count", "ngram"],
            ascending=[False, True]
        )
        .head(top_n)
        .reset_index(drop=True)
    )

In [ ]:

unigram_tables = []
bigram_tables = []

for group in group_order:
    unigram_tables.append(
        get_group_ngrams(
            mpnet_error_df,
            group=group,
            ngram_range=(1, 1),
            top_n=20
        )
    )

    bigram_tables.append(
        get_group_ngrams(
            mpnet_error_df,
            group=group,
            ngram_range=(2, 2),
            top_n=20
        )
    )

top_unigrams = pd.concat(
    unigram_tables,
    ignore_index=True
)

top_bigrams = pd.concat(
    bigram_tables,
    ignore_index=True
)

for group in group_order:
    print("=" * 80)
    print(f"TOP UNIGRAMS: {group}")
    display(
        top_unigrams.loc[
            top_unigrams["prediction_group"] == group,
            ["ngram", "count", "per_1000_ngrams"]
        ].reset_index(drop=True)
    )

    print(f"TOP BIGRAMS: {group}")
    display(
        top_bigrams.loc[
            top_bigrams["prediction_group"] == group,
            ["ngram", "count", "per_1000_ngrams"]
        ].reset_index(drop=True)
    )

top_unigrams.to_csv(
    "/content/mpnet_top_unigrams.csv",
    index=False
)

top_bigrams.to_csv(
    "/content/mpnet_top_bigrams.csv",
    index=False
)

print("Saved unigram and bigram tables.")

In [ ]:

def pivot_ngram_results(ngram_dataframe, value_column):
    return (
        ngram_dataframe
        .pivot(
            index="ngram",
            columns="prediction_group",
            values=value_column
        )
        .reindex(columns=group_order)
        .fillna(0)
    )

unigram_count_comparison = pivot_ngram_results(
    top_unigrams,
    "count"
)

unigram_frequency_comparison = pivot_ngram_results(
    top_unigrams,
    "per_1000_ngrams"
)

bigram_count_comparison = pivot_ngram_results(
    top_bigrams,
    "count"
)

bigram_frequency_comparison = pivot_ngram_results(
    top_bigrams,
    "per_1000_ngrams"
)

print("Top-unigram raw counts:")
display(unigram_count_comparison)

print("\nTop-unigram frequencies per 1,000 unigrams:")
display(unigram_frequency_comparison.round(3))

print("\nTop-bigram raw counts:")
display(bigram_count_comparison)

print("\nTop-bigram frequencies per 1,000 bigrams:")
display(bigram_frequency_comparison.round(3))

In [ ]:

target_patterns = {
    "worried": r"\bworried\b",
    "tired": r"\btired\b",
    "afraid": r"\bafraid\b",
    "alone": r"\balone\b",
    "sad": r"\bsad\b",
    "can't / cannot": r"\b(?:can't|cannot|cant)\b"
}

target_word_rows = []

for group in group_order:
    group_texts = (
        mpnet_error_df.loc[
            mpnet_error_df["prediction_group"] == group,
            "text"
        ]
        .fillna("")
        .astype(str)
    )

    group_word_total = group_texts.apply(count_words).sum()

    for term_name, pattern in target_patterns.items():
        occurrence_count = int(
            group_texts
            .str.count(
                pattern,
                flags=re.IGNORECASE
            )
            .sum()
        )

        posts_containing_term = int(
            group_texts
            .str.contains(
                pattern,
                case=False,
                regex=True,
                na=False
            )
            .sum()
        )

        target_word_rows.append({
            "prediction_group": group,
            "term": term_name,
            "occurrence_count": occurrence_count,
            "posts_containing_term": posts_containing_term,
            "percent_of_group_posts": (
                posts_containing_term / len(group_texts) * 100
                if len(group_texts) > 0
                else 0
            ),
            "occurrences_per_1000_words": (
                occurrence_count / group_word_total * 1000
                if group_word_total > 0
                else 0
            )
        })

target_word_results = pd.DataFrame(target_word_rows)

target_word_results["prediction_group"] = pd.Categorical(
    target_word_results["prediction_group"],
    categories=group_order,
    ordered=True
)

target_word_results = target_word_results.sort_values(
    ["term", "prediction_group"]
).reset_index(drop=True)

display(target_word_results.round(3))

print("Occurrences per 1,000 words:")
display(
    target_word_results.pivot(
        index="term",
        columns="prediction_group",
        values="occurrences_per_1000_words"
    ).reindex(columns=group_order).round(3)
)

target_word_results.to_csv(
    "/content/mpnet_target_word_comparison.csv",
    index=False
)

### LIME Explanations

Generate local explanations for 10 false positives, 10 false negatives,
5 true positives, and 5 true negatives.


In [ ]:
from pathlib import Path
from lime.lime_text import LimeTextExplainer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

mpnet_output_directory = Path(
    "/content/MPNet_full_fine_tuning"
)

checkpoint_directories = list(
    mpnet_output_directory.glob("checkpoint-*")
)

if not checkpoint_directories:
    raise FileNotFoundError(
        "No saved MPNet checkpoint was found. Make sure that "
        "KEEP_CHECKPOINTS = True, and then rerun the MPNet "
        "full-fine-tuning cell."
    )

def checkpoint_number(path):
    try:
        return int(path.name.split("-")[-1])
    except ValueError:
        return -1

mpnet_checkpoint_path = max(
    checkpoint_directories,
    key=checkpoint_number
)

print("Loading trained MPNet checkpoint:")
print(mpnet_checkpoint_path)

lime_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINTS["MPNet"],
    use_fast=True
)

lime_model = AutoModelForSequenceClassification.from_pretrained(
    str(mpnet_checkpoint_path)
)

lime_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

lime_model.to(lime_device)
lime_model.eval()

print("LIME model device:", lime_device)

In [ ]:
def mpnet_predict_proba(texts, batch_size=16):
    """
    Return an N x 2 array:
    column 0 = P(non-stress)
    column 1 = P(stress)
    """
    texts = [str(text) for text in texts]
    probability_batches = []

    for start_index in range(0, len(texts), batch_size):
        batch_texts = texts[
            start_index:start_index + batch_size
        ]

        encoded_batch = lime_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded_batch = {
            key: value.to(lime_device)
            for key, value in encoded_batch.items()
        }

        with torch.no_grad():
            logits = lime_model(
                **encoded_batch
            ).logits

            batch_probabilities = torch.softmax(
                logits,
                dim=1
            ).cpu().numpy()

        probability_batches.append(batch_probabilities)

    return np.vstack(probability_batches)

reloaded_test_probabilities = mpnet_predict_proba(
    transformer_test_df["text"].tolist()
)

reloaded_predictions = np.argmax(
    reloaded_test_probabilities,
    axis=1
)

agreement = np.mean(
    reloaded_predictions
    == mpnet_error_df["predicted_label"].to_numpy()
)

maximum_probability_difference = np.max(
    np.abs(
        reloaded_test_probabilities[:, 1]
        - mpnet_error_df["probability_stress"].to_numpy()
    )
)

print(f"Prediction agreement: {agreement:.3%}")
print(
    "Maximum stress-probability difference:",
    f"{maximum_probability_difference:.8f}"
)

if agreement < 0.99:
    raise RuntimeError(
        "The reloaded checkpoint does not sufficiently reproduce "
        "the earlier MPNet predictions. Do not continue to LIME "
        "until the checkpoint issue is resolved."
    )

In [ ]:

lime_sample_sizes = {
    "FP": 10,
    "FN": 10,
    "TP": 5,
    "TN": 5
}

lime_samples = []

for group, sample_size in lime_sample_sizes.items():
    available_examples = mpnet_error_df.loc[
        mpnet_error_df["prediction_group"] == group
    ]

    if len(available_examples) < sample_size:
        raise ValueError(
            f"Only {len(available_examples)} {group} examples "
            f"are available, but {sample_size} were requested."
        )

    sampled_examples = available_examples.sample(
        n=sample_size,
        random_state=SEED
    )

    lime_samples.append(sampled_examples)

lime_sample_df = (
    pd.concat(lime_samples)
    .reset_index()
    .rename(columns={"index": "original_test_index"})
)

lime_sample_df["lime_example_number"] = (
    lime_sample_df
    .groupby("prediction_group")
    .cumcount() + 1
)

display(
    lime_sample_df[
        [
            "original_test_index",
            "prediction_group",
            "lime_example_number",
            "true_label",
            "predicted_label",
            "probability_stress",
            "text"
        ]
    ]
)

print("\nSelected sample counts:")
display(
    lime_sample_df["prediction_group"]
    .value_counts()
    .reindex(group_order)
)

In [ ]:

false_positives = (
    mpnet_error_df[
        mpnet_error_df["prediction_group"] == "FP"
    ]
    .sort_values(
        "probability_stress",
        ascending=False
    )
    .reset_index(drop=True)
)

print(f"Total false positives: {len(false_positives)}")

for index, row in false_positives.iterrows():
    print("=" * 100)
    print(f"FALSE POSITIVE {index + 1}")
    print(f"ID: {row['id']}")
    print(f"Subreddit: {row['subreddit']}")
    print(f"True label: {row['true_label_name']}")
    print(f"Predicted label: {row['predicted_label_name']}")
    print(f"Stress probability: {row['probability_stress']:.4f}")
    print(f"Prediction confidence: {row['prediction_confidence']:.4f}")
    print("\nTEXT:")
    print(row["text"])
    print()

In [ ]:
import os

lime_explainer = LimeTextExplainer(
    class_names=["Non-stress", "Stress"],
    random_state=SEED
)

lime_output_directory = Path(
    "/content/mpnet_lime_explanations"
)

lime_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

lime_feature_rows = []
lime_example_rows = []

for row_number, row in lime_sample_df.iterrows():
    group = row["prediction_group"]
    example_number = int(row["lime_example_number"])
    text = str(row["text"])

    print(
        f"Explaining {group} example "
        f"{example_number}/{lime_sample_sizes[group]}"
    )

    explanation = lime_explainer.explain_instance(
        text_instance=text,
        classifier_fn=mpnet_predict_proba,
        labels=[1],
        num_features=15,
        num_samples=1000
    )

    feature_weights = explanation.as_list(label=1)

    stress_features = [
        feature
        for feature, weight in feature_weights
        if weight > 0
    ]

    nonstress_features = [
        feature
        for feature, weight in feature_weights
        if weight < 0
    ]

    html_filename = (
        f"{group}_{example_number:02d}"
        f"_testindex_{int(row['original_test_index'])}.html"
    )

    html_path = lime_output_directory / html_filename

    explanation.save_to_file(
        str(html_path),
        labels=[1]
    )

    lime_example_rows.append({
        "original_test_index": int(
            row["original_test_index"]
        ),
        "prediction_group": group,
        "lime_example_number": example_number,
        "true_label": int(row["true_label"]),
        "predicted_label": int(row["predicted_label"]),
        "probability_nonstress": float(
            row["probability_nonstress"]
        ),
        "probability_stress": float(
            row["probability_stress"]
        ),
        "prediction_confidence": float(
            row["prediction_confidence"]
        ),
        "stress_pushing_features": ", ".join(
            stress_features
        ),
        "nonstress_pushing_features": ", ".join(
            nonstress_features
        ),
        "text": text,
        "lime_html_file": str(html_path)
    })

    for feature_rank, (feature, weight) in enumerate(
        feature_weights,
        start=1
    ):
        lime_feature_rows.append({
            "original_test_index": int(
                row["original_test_index"]
            ),
            "prediction_group": group,
            "lime_example_number": example_number,
            "feature_rank": feature_rank,
            "feature": feature,
            "weight_toward_stress": float(weight),
            "direction": (
                "Toward stress"
                if weight > 0
                else "Toward non-stress"
            )
        })

lime_example_summary = pd.DataFrame(lime_example_rows)
lime_feature_details = pd.DataFrame(lime_feature_rows)

print("LIME explanations complete.")

In [ ]:

lime_summary_columns = [
    "prediction_group",
    "lime_example_number",
    "true_label",
    "predicted_label",
    "probability_stress",
    "prediction_confidence",
    "stress_pushing_features",
    "nonstress_pushing_features",
    "text"
]

for group in group_order:
    print("=" * 100)
    print(f"LIME EXPLANATIONS: {group}")

    group_summary = lime_example_summary.loc[
        lime_example_summary["prediction_group"] == group,
        lime_summary_columns
    ]

    display(group_summary.reset_index(drop=True))

lime_example_summary.to_csv(
    "/content/mpnet_lime_example_summary.csv",
    index=False
)

lime_feature_details.to_csv(
    "/content/mpnet_lime_feature_details.csv",
    index=False
)

print("Saved:")
print("/content/mpnet_lime_example_summary.csv")
print("/content/mpnet_lime_feature_details.csv")

In [ ]:

stress_pushing_features = lime_feature_details.loc[
    lime_feature_details["weight_toward_stress"] > 0
].copy()

aggregated_stress_features = (
    stress_pushing_features
    .groupby(["prediction_group", "feature"])
    .agg(
        number_of_explanations=(
            "original_test_index",
            "nunique"
        ),
        mean_positive_weight=(
            "weight_toward_stress",
            "mean"
        ),
        maximum_positive_weight=(
            "weight_toward_stress",
            "max"
        )
    )
    .reset_index()
)

aggregated_stress_features["prediction_group"] = pd.Categorical(
    aggregated_stress_features["prediction_group"],
    categories=group_order,
    ordered=True
)

aggregated_stress_features = (
    aggregated_stress_features
    .sort_values(
        [
            "prediction_group",
            "number_of_explanations",
            "mean_positive_weight"
        ],
        ascending=[True, False, False]
    )
)

for group in group_order:
    print("=" * 80)
    print(f"REPEATED STRESS-PUSHING FEATURES: {group}")

    display(
        aggregated_stress_features.loc[
            aggregated_stress_features[
                "prediction_group"
            ] == group
        ].head(20).reset_index(drop=True)
    )

aggregated_stress_features.to_csv(
    "/content/mpnet_lime_aggregated_stress_features.csv",
    index=False
)

In [ ]:

import shutil

lime_zip_path = shutil.make_archive(
    "/content/mpnet_lime_explanations",
    "zip",
    root_dir=str(lime_output_directory)
)

print("Created:", lime_zip_path)

In [ ]:

mpnet_error_df.to_csv(
    "/content/mpnet_error_analysis_master.csv",
    index=False
)

print("Main output files:")
print("1. /content/mpnet_error_analysis_master.csv")
print("2. /content/mpnet_task2_group_means.csv")
print("3. /content/mpnet_task2_group_medians.csv")
print("4. /content/mpnet_task2_focused_comparisons.csv")
print("5. /content/mpnet_top_unigrams.csv")
print("6. /content/mpnet_top_bigrams.csv")
print("7. /content/mpnet_target_word_comparison.csv")
print("8. /content/mpnet_lime_example_summary.csv")
print("9. /content/mpnet_lime_feature_details.csv")
print("10. /content/mpnet_lime_aggregated_stress_features.csv")
print("11. /content/mpnet_lime_explanations.zip")

# LoRA Fine-Tuning Experiments

The LoRA experiments below reuse the same cleaned dataset, the same canonical train / validation / official-test rows, and the same tokenized datasets created above. The pretrained transformer setup and full-fine-tuning runs are not repeated.


In [ ]:
LORA_CONFIGS = {
    "LoRA-small": {
        "r": 4,
        "lora_alpha": 8,
        "lora_dropout": 0.1,
    },
    "LoRA-medium": {
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.1,
    },
    "LoRA-large": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
    },
}

LORA_LEARNING_RATE = 2e-4
LORA_BATCH_SIZE = 8
LORA_WEIGHT_DECAY = 0.01

lora_results = []
lora_predictions = {}

print("Models reused from the transformer section:")
for name, checkpoint in MODEL_CHECKPOINTS.items():
    print(f"  {name}: {checkpoint}")

print("\nLoRA configurations:")
for name, config in LORA_CONFIGS.items():
    print(f"  {name}: {config}")


## 7. LoRA model configuration and training function

In [ ]:
def get_lora_model_settings(model):
    model_type = model.config.model_type

    if model_type == "bert":
        return {
            "target_modules": ["query", "value"],
            "modules_to_save": ["classifier"],
        }

    if model_type == "mpnet":
        return {
            "target_modules": ["q", "v"],
            "modules_to_save": ["classifier"],
        }

    if model_type == "distilbert":
        return {
            "target_modules": ["q_lin", "v_lin"],
            "modules_to_save": ["pre_classifier", "classifier"],
        }

    raise ValueError(
        f"No LoRA settings have been defined for model type: {model_type}"
    )


def run_lora_experiment(
    model_short_name,
    checkpoint,
    lora_setting_name,
):
    experiment_name = (
        f"{model_short_name}_"
        f"{lora_setting_name.replace('-', '_')}"
    )
    output_directory = f"/content/{experiment_name}"
    shutil.rmtree(output_directory, ignore_errors=True)

    lora_settings = LORA_CONFIGS[lora_setting_name]

    print("=" * 90)
    print("Experiment:", experiment_name)
    print("Checkpoint:", checkpoint)
    print("LoRA setting:", lora_setting_name)
    print("=" * 90)

    tokenizer, tokenized_datasets = get_tokenized_datasets(
        checkpoint
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=2,
        id2label={
            0: "Non-stress",
            1: "Stress/Stress-related",
        },
        label2id={
            "Non-stress": 0,
            "Stress/Stress-related": 1,
        },
    )

    architecture_settings = get_lora_model_settings(model)

    print("Model type:", model.config.model_type)
    print(
        "LoRA target modules:",
        architecture_settings["target_modules"],
    )
    print(
        "Classification modules kept trainable:",
        architecture_settings["modules_to_save"],
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=lora_settings["r"],
        lora_alpha=lora_settings["lora_alpha"],
        lora_dropout=lora_settings["lora_dropout"],
        bias="none",
        target_modules=architecture_settings["target_modules"],
        modules_to_save=architecture_settings["modules_to_save"],
    )

    model = get_peft_model(model, lora_config)

    print("\nPEFT parameter report:")
    model.print_trainable_parameters()

    trainable_parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    total_parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_percentage = (
        100.0
        * trainable_parameter_count
        / total_parameter_count
    )

    print(
        f"Trainable parameters: {trainable_parameter_count:,}"
    )
    print(
        f"Total parameters: {total_parameter_count:,}"
    )
    print(
        f"Percentage trainable: {trainable_percentage:.4f}%"
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8
        if torch.cuda.is_available()
        else None,
    )

    training_arguments = TrainingArguments(
        output_dir=output_directory,
        learning_rate=LORA_LEARNING_RATE,
        per_device_train_batch_size=LORA_BATCH_SIZE,
        per_device_eval_batch_size=16,
        num_train_epochs=NUMBER_OF_EPOCHS,
        weight_decay=LORA_WEIGHT_DECAY,
        warmup_steps=int(np.ceil(0.10 * NUMBER_OF_EPOCHS * np.ceil(
            len(tokenized_datasets["train"]) / LORA_BATCH_SIZE
        ))),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        optim="adamw_torch",
        seed=SEED,
        data_seed=SEED,
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_trainer_metrics,
    )

    trainer.train()

    validation_output = trainer.predict(
        tokenized_datasets["validation"]
    )

    validation_logits = validation_output.predictions
    if isinstance(validation_logits, tuple):
        validation_logits = validation_logits[0]

    validation_labels = validation_output.label_ids
    validation_predictions = np.argmax(
        validation_logits,
        axis=1
    )
    validation_probabilities = softmax_numpy(
        validation_logits
    )[:, 1]

    test_output = trainer.predict(
        tokenized_datasets["test"]
    )

    test_logits = test_output.predictions
    if isinstance(test_logits, tuple):
        test_logits = test_logits[0]

    test_labels = test_output.label_ids
    test_predictions = np.argmax(
        test_logits,
        axis=1,
    )
    test_probabilities = softmax_numpy(
        test_logits
    )[:, 1]

    test_metrics = calculate_binary_metrics(
        test_labels,
        test_predictions,
        test_probabilities,
    )

    best_validation_f1 = trainer.state.best_metric

    experiment_result = {
        "model": model_short_name,
        "checkpoint": checkpoint,
        "training_method": lora_setting_name,
        "r": lora_settings["r"],
        "lora_alpha": lora_settings["lora_alpha"],
        "lora_dropout": lora_settings["lora_dropout"],
        "learning_rate": LORA_LEARNING_RATE,
        "train_batch_size": LORA_BATCH_SIZE,
        "epochs": NUMBER_OF_EPOCHS,
        "max_length": MAX_LENGTH,
        "total_parameters": total_parameter_count,
        "trainable_parameters": trainable_parameter_count,
        "trainable_percentage": trainable_percentage,
        "best_validation_f1": (
            float(best_validation_f1)
            if best_validation_f1 is not None
            else np.nan
        ),
        **test_metrics,
    }

    lora_results[:] = [
        existing
        for existing in lora_results
        if not (
            existing["model"] == model_short_name
            and existing["training_method"] == lora_setting_name
        )
    ]
    lora_results.append(experiment_result)

    lora_predictions[experiment_name] = {
        "validation_labels": validation_labels,
        "validation_predictions": validation_predictions,
        "validation_probabilities": validation_probabilities,
        "validation_logits": validation_logits,
        "labels": test_labels,
        "predictions": test_predictions,
        "probabilities": test_probabilities,
        "test_logits": test_logits,
    }

    print("\nOfficial test-set results:")
    display(pd.DataFrame([experiment_result]))

    print("\nClassification report:")
    print(
        classification_report(
            test_labels,
            test_predictions,
            labels=[0, 1],
            target_names=[
                "Non-stress",
                "Stress/Stress-related",
            ],
            digits=3,
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        test_labels,
        test_predictions,
        labels=[0, 1],
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-stress", "Stress"],
    ).plot()

    plt.title(
        f"Confusion Matrix: {experiment_name}"
    )
    plt.show()

    RocCurveDisplay.from_predictions(
        test_labels,
        test_probabilities,
        name=experiment_name,
    )

    plt.title(
        f"ROC Curve: {experiment_name}"
    )
    plt.show()

    shutil.rmtree(
        output_directory,
        ignore_errors=True,
    )

    del trainer
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return experiment_result

## Nine LoRA Configurations

Run small, medium, and large LoRA configurations for DistilBERT, MiniLM,
and MPNet.


In [ ]:
EXPERIMENT_ORDER = [
    "DistilBERT",
    "MiniLM",
    "MPNet",
]

for model_name in EXPERIMENT_ORDER:
    checkpoint = MODEL_CHECKPOINTS[model_name]

    for lora_setting_name in [
        "LoRA-small",
        "LoRA-medium",
        "LoRA-large",
    ]:
        run_lora_experiment(
            model_short_name=model_name,
            checkpoint=checkpoint,
            lora_setting_name=lora_setting_name,
        )

lora_results_df = pd.DataFrame(lora_results)

print("\nAll LoRA experiments completed.")
display(
    lora_results_df.sort_values(
        ["model", "r"]
    ).reset_index(drop=True)
)

## Full-fine-tuning baselines from the runs above

The comparison below pulls the full-fine-tuning results directly from `transformer_results`, so the three full-fine-tuning models are not trained a second time.


In [ ]:
full_finetuning_baselines = pd.DataFrame([
    result
    for result in transformer_results
    if result["training_method"] == "full_fine_tuning"
]).copy()

if len(full_finetuning_baselines) != 3:
    raise RuntimeError(
        "Expected three full-fine-tuning results from the earlier section. "
        "Run the MiniLM, MPNet, and DistilBERT full-fine-tuning cells before this comparison."
    )

full_finetuning_baselines["training_method"] = "Full fine-tuning"
full_finetuning_baselines["r"] = np.nan
full_finetuning_baselines["lora_alpha"] = np.nan
full_finetuning_baselines["lora_dropout"] = np.nan
full_finetuning_baselines["trainable_percentage"] = (
    100.0
    * full_finetuning_baselines["trainable_parameters"]
    / full_finetuning_baselines["total_parameters"]
)

display(full_finetuning_baselines)


## 10. Final LoRA vs. full-fine-tuning comparison

In [ ]:
comparison_df = pd.concat(
    [
        full_finetuning_baselines,
        lora_results_df,
    ],
    ignore_index=True,
)

method_order = {
    "Full fine-tuning": 0,
    "LoRA-small": 1,
    "LoRA-medium": 2,
    "LoRA-large": 3,
}

comparison_df["_method_order"] = (
    comparison_df["training_method"].map(method_order)
)

comparison_df = (
    comparison_df
    .sort_values(["model", "_method_order"])
    .drop(columns=["_method_order"])
    .reset_index(drop=True)
)

comparison_columns = [
    "model",
    "training_method",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "tn",
    "fp",
    "fn",
    "tp",
    "total_parameters",
    "trainable_parameters",
    "trainable_percentage",
]

display(comparison_df[comparison_columns])

comparison_df.to_csv(
    "/content/lora_vs_full_finetuning_results.csv",
    index=False,
)

print(
    "Saved: /content/lora_vs_full_finetuning_results.csv"
)

## 11. Parameter-efficiency comparison

In [ ]:
full_ft_reference = (
    full_finetuning_baselines
    .set_index("model")
)

efficiency_rows = []

for _, row in lora_results_df.iterrows():
    model_name = row["model"]
    full_ft = full_ft_reference.loc[model_name]

    efficiency_rows.append({
        "model": model_name,
        "LoRA_setting": row["training_method"],
        "full_FT_f1": full_ft["f1"],
        "LoRA_f1": row["f1"],
        "f1_difference": row["f1"] - full_ft["f1"],
        "full_FT_accuracy": full_ft["accuracy"],
        "LoRA_accuracy": row["accuracy"],
        "accuracy_difference": (
            row["accuracy"]
            - full_ft["accuracy"]
        ),
        "full_FT_trainable_parameters": (
            full_ft["trainable_parameters"]
        ),
        "LoRA_trainable_parameters": (
            row["trainable_parameters"]
        ),
        "LoRA_percentage_trainable": (
            row["trainable_percentage"]
        ),
        "parameter_reduction_percent": (
            100.0
            * (
                1.0
                - row["trainable_parameters"]
                / full_ft["trainable_parameters"]
            )
        ),
    })

efficiency_comparison_df = pd.DataFrame(
    efficiency_rows
).sort_values(
    ["model", "LoRA_setting"]
).reset_index(drop=True)

display(efficiency_comparison_df)

efficiency_comparison_df.to_csv(
    "/content/lora_parameter_efficiency_comparison.csv",
    index=False,
)

print(
    "Saved: /content/lora_parameter_efficiency_comparison.csv"
)

## 12. Quick visual comparison

In [ ]:
plot_df = comparison_df[
    ["model", "training_method", "f1", "trainable_percentage"]
].copy()

for model_name in MODEL_CHECKPOINTS:
    model_plot = plot_df[
        plot_df["model"] == model_name
    ].copy()

    plt.figure(figsize=(8, 4))
    plt.bar(
        model_plot["training_method"],
        model_plot["f1"],
    )
    plt.title(
        f"{model_name}: F1 Score by Training Method"
    )
    plt.ylabel("F1 Score")
    plt.ylim(0, 1)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

for model_name in MODEL_CHECKPOINTS:
    model_plot = plot_df[
        plot_df["model"] == model_name
    ].copy()

    plt.figure(figsize=(8, 4))
    plt.bar(
        model_plot["training_method"],
        model_plot["trainable_percentage"],
    )
    plt.title(
        f"{model_name}: Percentage of Parameters Trained"
    )
    plt.ylabel("Trainable Parameters (%)")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

## Original LoRA Comparison Outputs

The original analysis saves `lora_vs_full_finetuning_results.csv` and
`lora_parameter_efficiency_comparison.csv`.


# Uncertainty, Calibration, and Abstention

Temperature scaling is fitted using the shared validation split, and final
calibration is evaluated on the official test split. Models are selected by
validation F1.


## Select Models Using Validation Scores

Reuse the best previously trained full-fine-tuning and LoRA models without
additional training.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

standard_candidates = (
    pd.DataFrame(transformer_results)
    .query("training_method == 'full_fine_tuning'")
    .sort_values("best_validation_f1", ascending=False)
    .reset_index(drop=True)
)

lora_candidates = (
    pd.DataFrame(lora_results)
    .sort_values("best_validation_f1", ascending=False)
    .reset_index(drop=True)
)

if standard_candidates.empty:
    raise RuntimeError(
        "No full-fine-tuning results found. Run the transformer experiments above first."
    )

if lora_candidates.empty:
    raise RuntimeError(
        "No LoRA results found. Run the LoRA experiments above first."
    )

best_standard_row = standard_candidates.iloc[0]
best_lora_row = lora_candidates.iloc[0]

BEST_STANDARD_KEY = (
    f"{best_standard_row['model']}_full_fine_tuning"
)

BEST_LORA_KEY = (
    f"{best_lora_row['model']}_"
    f"{best_lora_row['training_method'].replace('-', '_')}"
)

print("Best standard fine-tuned model:")
display(best_standard_row.to_frame().T)

print("\nBest LoRA model:")
display(best_lora_row.to_frame().T)

print("\nPrediction stores:")
print("Standard:", BEST_STANDARD_KEY)
print("LoRA:", BEST_LORA_KEY)


## Prediction Probabilities and Confidence

Store labels, predicted stress probabilities, confidence, and error groups
for official test samples.


In [ ]:
def prediction_group(true_label, predicted_label):
    if true_label == 1 and predicted_label == 1:
        return "TP"
    if true_label == 0 and predicted_label == 0:
        return "TN"
    if true_label == 0 and predicted_label == 1:
        return "FP"
    return "FN"


def make_uncertainty_dataframe(model_label, stored_predictions):
    labels = np.asarray(stored_predictions["labels"]).astype(int)
    probabilities = np.asarray(
        stored_predictions["probabilities"]
    ).astype(float)

    predictions = (probabilities >= 0.5).astype(int)
    confidence = np.maximum(
        probabilities,
        1.0 - probabilities
    )

    dataframe = pd.DataFrame({
        "model": model_label,
        "true_label": labels,
        "predicted_label": predictions,
        "probability_stress": probabilities,
        "model_confidence": confidence,
    })

    dataframe["prediction_group"] = [
        prediction_group(y, p)
        for y, p in zip(
            dataframe["true_label"],
            dataframe["predicted_label"]
        )
    ]

    return dataframe


standard_uncertainty_df = make_uncertainty_dataframe(
    f"{best_standard_row['model']} Full Fine-Tuning",
    transformer_predictions[BEST_STANDARD_KEY]
)

lora_uncertainty_df = make_uncertainty_dataframe(
    f"{best_lora_row['model']} {best_lora_row['training_method']}",
    lora_predictions[BEST_LORA_KEY]
)

uncertainty_predictions_df = pd.concat(
    [standard_uncertainty_df, lora_uncertainty_df],
    ignore_index=True
)

display(uncertainty_predictions_df.head())

standard_uncertainty_df.to_csv(
    "/content/best_standard_uncertainty_predictions.csv",
    index=False
)

lora_uncertainty_df.to_csv(
    "/content/best_lora_uncertainty_predictions.csv",
    index=False
)


## Confidence by Error Type

Compare the confidence distributions of TP, TN, FP, and FN predictions.


In [ ]:
group_order = ["TP", "TN", "FP", "FN"]

confidence_by_error_type = (
    uncertainty_predictions_df
    .groupby(["model", "prediction_group"])
    .agg(
        number_of_examples=("true_label", "size"),
        average_confidence=("model_confidence", "mean"),
        median_confidence=("model_confidence", "median"),
        average_probability_stress=(
            "probability_stress",
            "mean"
        ),
    )
    .reset_index()
)

confidence_by_error_type["prediction_group"] = pd.Categorical(
    confidence_by_error_type["prediction_group"],
    categories=group_order,
    ordered=True
)

confidence_by_error_type = (
    confidence_by_error_type
    .sort_values(["model", "prediction_group"])
    .reset_index(drop=True)
)

display(confidence_by_error_type)

confidence_by_error_type.to_csv(
    "/content/confidence_by_error_type.csv",
    index=False
)


## Calibration Curves and Metrics

Reliability diagrams and expected calibration error compare predicted
confidence with empirical accuracy.


In [ ]:
def geometric_mean_score_binary(labels, predictions):
    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    return float(np.sqrt(sensitivity * specificity))


def expected_calibration_error(
    labels,
    probabilities_stress,
    n_bins=10
):
    labels = np.asarray(labels).astype(int)
    probabilities_stress = np.asarray(
        probabilities_stress
    ).astype(float)

    predictions = (
        probabilities_stress >= 0.5
    ).astype(int)

    confidence = np.maximum(
        probabilities_stress,
        1.0 - probabilities_stress
    )

    correctness = (
        predictions == labels
    ).astype(float)

    bin_edges = np.linspace(
        0.5,
        1.0,
        n_bins + 1
    )

    ece = 0.0
    bin_rows = []

    for bin_index in range(n_bins):
        lower = bin_edges[bin_index]
        upper = bin_edges[bin_index + 1]

        if bin_index == n_bins - 1:
            in_bin = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            in_bin = (
                (confidence >= lower)
                & (confidence < upper)
            )

        count = int(in_bin.sum())

        if count == 0:
            continue

        average_confidence = float(
            confidence[in_bin].mean()
        )
        average_accuracy = float(
            correctness[in_bin].mean()
        )
        weight = count / len(labels)

        ece += weight * abs(
            average_confidence
            - average_accuracy
        )

        bin_rows.append({
            "lower_confidence": lower,
            "upper_confidence": upper,
            "count": count,
            "average_confidence": average_confidence,
            "average_accuracy": average_accuracy,
            "absolute_gap": abs(
                average_confidence
                - average_accuracy
            ),
        })

    return float(ece), pd.DataFrame(bin_rows)


def calibration_metrics(
    labels,
    probabilities_stress
):
    labels = np.asarray(labels).astype(int)
    probabilities_stress = np.asarray(
        probabilities_stress
    ).astype(float)

    predictions = (
        probabilities_stress >= 0.5
    ).astype(int)

    ece, _ = expected_calibration_error(
        labels,
        probabilities_stress,
        n_bins=10
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "g_mean": geometric_mean_score_binary(
            labels,
            predictions
        ),
        "auc": roc_auc_score(
            labels,
            probabilities_stress
        ),
        "brier_score": brier_score_loss(
            labels,
            probabilities_stress
        ),
        "ece": ece,
    }


model_probability_sets = {
    f"{best_standard_row['model']} Full Fine-Tuning": (
        transformer_predictions[BEST_STANDARD_KEY]["labels"],
        transformer_predictions[BEST_STANDARD_KEY]["probabilities"],
    ),
    f"{best_lora_row['model']} {best_lora_row['training_method']}": (
        lora_predictions[BEST_LORA_KEY]["labels"],
        lora_predictions[BEST_LORA_KEY]["probabilities"],
    ),
}

original_metric_rows = []

for model_name, (labels, probabilities) in model_probability_sets.items():
    metrics = calibration_metrics(
        labels,
        probabilities
    )
    metrics["model"] = model_name
    original_metric_rows.append(metrics)

original_calibration_metrics_df = pd.DataFrame(
    original_metric_rows
)[
    [
        "model",
        "accuracy",
        "g_mean",
        "auc",
        "brier_score",
        "ece",
    ]
]

display(original_calibration_metrics_df)


In [ ]:
for model_name, (labels, probabilities) in model_probability_sets.items():
    fraction_positives, mean_predicted_probability = (
        calibration_curve(
            labels,
            probabilities,
            n_bins=10,
            strategy="uniform"
        )
    )

    plt.figure(figsize=(6, 6))
    plt.plot(
        mean_predicted_probability,
        fraction_positives,
        marker="o",
        label=model_name
    )
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )
    plt.xlabel("Mean predicted probability of stress")
    plt.ylabel("Fraction actually stressed")
    plt.title(f"Reliability Diagram: {model_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Temperature Scaling

Fit one positive temperature on validation logits and apply it to the test
logits. The scaling changes confidence but not class predictions.


In [ ]:
class TemperatureScaler(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = torch.nn.Parameter(
            torch.zeros(1)
        )

    @property
    def temperature(self):
        return torch.exp(self.log_temperature)

    def forward(self, logits):
        return logits / self.temperature

    def fit(self, validation_logits, validation_labels):
        logits_tensor = torch.tensor(
            validation_logits,
            dtype=torch.float32
        )
        labels_tensor = torch.tensor(
            validation_labels,
            dtype=torch.long
        )

        criterion = torch.nn.CrossEntropyLoss()

        optimizer = torch.optim.LBFGS(
            [self.log_temperature],
            lr=0.1,
            max_iter=100,
            line_search_fn="strong_wolfe"
        )

        def closure():
            optimizer.zero_grad()
            loss = criterion(
                self.forward(logits_tensor),
                labels_tensor
            )
            loss.backward()
            return loss

        optimizer.step(closure)

        return float(
            self.temperature.detach().cpu().item()
        )


def temperature_scaled_probabilities(
    validation_logits,
    validation_labels,
    test_logits
):
    scaler = TemperatureScaler()

    temperature = scaler.fit(
        validation_logits,
        validation_labels
    )

    scaled_test_logits = (
        np.asarray(test_logits)
        / temperature
    )

    scaled_test_probabilities = (
        softmax_numpy(scaled_test_logits)[:, 1]
    )

    return temperature, scaled_test_probabilities


selected_prediction_stores = {
    f"{best_standard_row['model']} Full Fine-Tuning":
        transformer_predictions[BEST_STANDARD_KEY],
    f"{best_lora_row['model']} {best_lora_row['training_method']}":
        lora_predictions[BEST_LORA_KEY],
}

calibrated_results = {}
temperature_rows = []

for model_name, stored in selected_prediction_stores.items():
    temperature, scaled_probabilities = (
        temperature_scaled_probabilities(
            stored["validation_logits"],
            stored["validation_labels"],
            stored["test_logits"]
        )
    )

    calibrated_results[model_name] = {
        "temperature": temperature,
        "original_probabilities": np.asarray(
            stored["probabilities"]
        ),
        "calibrated_probabilities": scaled_probabilities,
        "labels": np.asarray(stored["labels"]).astype(int),
    }

    temperature_rows.append({
        "model": model_name,
        "temperature": temperature,
    })

temperature_df = pd.DataFrame(temperature_rows)

display(temperature_df)


In [ ]:
calibration_comparison_rows = []

for model_name, result in calibrated_results.items():
    labels = result["labels"]

    original_metrics = calibration_metrics(
        labels,
        result["original_probabilities"]
    )

    calibrated_metrics = calibration_metrics(
        labels,
        result["calibrated_probabilities"]
    )

    calibration_comparison_rows.append({
        "model": model_name,
        "probabilities": "Original",
        "temperature": 1.0,
        **original_metrics,
    })

    calibration_comparison_rows.append({
        "model": model_name,
        "probabilities": "Temperature-scaled",
        "temperature": result["temperature"],
        **calibrated_metrics,
    })

calibration_comparison_df = pd.DataFrame(
    calibration_comparison_rows
)[
    [
        "model",
        "probabilities",
        "temperature",
        "accuracy",
        "g_mean",
        "auc",
        "brier_score",
        "ece",
    ]
]

display(calibration_comparison_df)

calibration_comparison_df.to_csv(
    "/content/temperature_scaling_metrics.csv",
    index=False
)


In [ ]:
for model_name, result in calibrated_results.items():
    labels = result["labels"]

    original_fraction, original_mean_probability = (
        calibration_curve(
            labels,
            result["original_probabilities"],
            n_bins=10,
            strategy="uniform"
        )
    )

    calibrated_fraction, calibrated_mean_probability = (
        calibration_curve(
            labels,
            result["calibrated_probabilities"],
            n_bins=10,
            strategy="uniform"
        )
    )

    plt.figure(figsize=(6, 6))
    plt.plot(
        original_mean_probability,
        original_fraction,
        marker="o",
        label="Original"
    )
    plt.plot(
        calibrated_mean_probability,
        calibrated_fraction,
        marker="o",
        label="Temperature-scaled"
    )
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )
    plt.xlabel("Mean predicted probability of stress")
    plt.ylabel("Fraction actually stressed")
    plt.title(
        f"Calibration Before vs. After Temperature Scaling\n{model_name}"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
ece_bin_tables = []

for model_name, result in calibrated_results.items():
    for version, probabilities in [
        ("Original", result["original_probabilities"]),
        ("Temperature-scaled", result["calibrated_probabilities"]),
    ]:
        ece_value, bins_df = expected_calibration_error(
            result["labels"],
            probabilities,
            n_bins=10
        )

        bins_df["model"] = model_name
        bins_df["probabilities"] = version
        bins_df["ece"] = ece_value

        ece_bin_tables.append(bins_df)

ece_bins_df = pd.concat(
    ece_bin_tables,
    ignore_index=True
)

display(ece_bins_df)

ece_bins_df.to_csv(
    "/content/ece_confidence_bins.csv",
    index=False
)


## Abstention Thresholds

Report test coverage and retained-sample performance at confidence thresholds
before and after temperature scaling.


In [ ]:
def abstention_results(
    model_name,
    labels,
    probabilities,
    probability_version,
    thresholds=(0.50, 0.60, 0.70, 0.80, 0.90)
):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(
        probabilities
    ).astype(float)

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    confidence = np.maximum(
        probabilities,
        1.0 - probabilities
    )

    rows = []

    for threshold in thresholds:
        keep = confidence >= threshold

        retained_labels = labels[keep]
        retained_predictions = predictions[keep]
        retained_probabilities = probabilities[keep]

        number_classified = int(keep.sum())
        number_abstained = int((~keep).sum())
        coverage = (
            number_classified / len(labels)
            if len(labels) > 0
            else np.nan
        )

        if number_classified == 0:
            accuracy = np.nan
            g_mean = np.nan
            auc = np.nan
            fp = fn = np.nan
        else:
            accuracy = accuracy_score(
                retained_labels,
                retained_predictions
            )

            g_mean = geometric_mean_score_binary(
                retained_labels,
                retained_predictions
            )

            if len(np.unique(retained_labels)) == 2:
                auc = roc_auc_score(
                    retained_labels,
                    retained_probabilities
                )
            else:
                auc = np.nan

            tn, fp, fn, tp = confusion_matrix(
                retained_labels,
                retained_predictions,
                labels=[0, 1]
            ).ravel()

        rows.append({
            "model": model_name,
            "probability_version": probability_version,
            "confidence_threshold": threshold,
            "number_classified": number_classified,
            "number_abstained": number_abstained,
            "coverage": coverage,
            "accuracy_on_classified": accuracy,
            "g_mean_on_classified": g_mean,
            "auc_on_classified": auc,
            "false_positives_on_classified": fp,
            "false_negatives_on_classified": fn,
        })

    return pd.DataFrame(rows)


abstention_tables = []

for model_name, result in calibrated_results.items():
    abstention_tables.append(
        abstention_results(
            model_name,
            result["labels"],
            result["original_probabilities"],
            "Original"
        )
    )

    abstention_tables.append(
        abstention_results(
            model_name,
            result["labels"],
            result["calibrated_probabilities"],
            "Temperature-scaled"
        )
    )

abstention_results_df = pd.concat(
    abstention_tables,
    ignore_index=True
)

display(abstention_results_df)

abstention_results_df.to_csv(
    "/content/abstention_threshold_results.csv",
    index=False
)


In [ ]:
for model_name in abstention_results_df["model"].unique():
    model_rows = abstention_results_df[
        abstention_results_df["model"] == model_name
    ]

    for probability_version in [
        "Original",
        "Temperature-scaled"
    ]:
        version_rows = model_rows[
            model_rows["probability_version"]
            == probability_version
        ]

        plt.figure(figsize=(7, 4))
        plt.plot(
            version_rows["confidence_threshold"],
            version_rows["accuracy_on_classified"],
            marker="o",
            label="Accuracy on classified examples"
        )
        plt.plot(
            version_rows["confidence_threshold"],
            version_rows["coverage"],
            marker="o",
            label="Coverage"
        )
        plt.xlabel("Confidence threshold")
        plt.ylabel("Proportion")
        plt.ylim(0, 1.02)
        plt.title(
            f"Abstention Tradeoff: {model_name}\n{probability_version}"
        )
        plt.legend()
        plt.tight_layout()
        plt.show()


## Calibration Outputs

The original calibration analysis saves prediction-level CSVs, confidence
summaries, ECE bins, and abstention results under `/content/`.


# Matched Ten-Seed Experiment

For each backbone and method, choose a learning rate using validation F1
without consulting test predictions. Both methods use effective batch size 8,
AdamW, 0.01 weight decay, 10% warmup, and up to five epochs. The best epoch
is selected by validation F1. The selected settings are retrained across
ten matched seeds and evaluated on the official test split.


In [ ]:
from pathlib import Path
import json, itertools, hashlib, gc, shutil
import numpy as np
import pandas as pd
import torch

REVIEW_DIR = Path('/content/reviewer_revision_outputs')
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

assert len(model_source_df) == 2817, f'Unexpected cleaned pool: {len(model_source_df)}'
assert len(model_train_df) == 2394 and len(model_val_df) == 423
assert len(model_test_df) == 715
assert set(model_train_df['text']).isdisjoint(set(model_test_df['text']))
assert set(model_val_df['text']).isdisjoint(set(model_test_df['text']))
assert set(model_train_df['text']).isdisjoint(set(model_val_df['text']))
assert model_test_df['label'].value_counts().to_dict() == {1: 369, 0: 346}

cleaning_audit = pd.DataFrame([{
    'official_training_rows': len(train_raw),
    'duplicate_training_rows_removed_by_text': int(duplicate_training_texts),
    'training_test_overlap_rows_removed': int(training_test_overlap_rows),
    'cleaned_training_pool': len(model_source_df),
    'training_rows': len(model_train_df),
    'validation_rows': len(model_val_df),
    'official_test_rows_unchanged': len(model_test_df),
}])
cleaning_audit.to_csv(REVIEW_DIR / 'reviewer_data_cleaning_audit.csv', index=False)
display(cleaning_audit)
assert int(duplicate_training_texts) == 18
assert int(training_test_overlap_rows) == 3
print('Clarification for manuscript: removed 18 repeated-text training rows, '
      'retaining one copy of each text, and removed 3 test-overlapping rows.')

## Published Dreaddit Baselines

The following values are taken from Turcan and McKeown (2019), Table 6,
https://aclanthology.org/D19-6213.pdf. They are reference values, not new
model runs, and training protocols differ.


In [ ]:
PUBLISHED_DREADDIT_BASELINES = [
    {'model': 'Majority baseline', 'precision': .5161, 'recall': 1.0000, 'f1': .6808},
    {'model': 'n-gram baseline', 'precision': .7249, 'recall': .7642, 'f1': .7441},
    {'model': 'n-grams + features', 'precision': .7474, 'recall': .7940, 'f1': .7700},
    {'model': 'LogReg + domain Word2Vec + features', 'precision': .7433,
     'recall': .8320, 'f1': .7980},
    {'model': 'BERT-base', 'precision': .7518, 'recall': .8699, 'f1': .8065},
]
published_df = pd.DataFrame(PUBLISHED_DREADDIT_BASELINES)
published_df['origin'] = 'Turcan and McKeown (2019), Table 6; published reference'
published_df['test_protocol'] = 'Published original protocol; not re-trained here'
published_df.to_csv(REVIEW_DIR / 'reviewer_published_dreaddit_baselines.csv', index=False)
display(published_df)

## LoRA Target Modules and MiniLM Diagnostics

LoRA adapts attention query/value projections and preserves trainable
classification heads. Other module subsets were not compared.
The MiniLM classifier-head-only analysis checks for near-single-class predictions.


In [ ]:
minilm_key = 'MiniLM_classifier_head_only'
if minilm_key in transformer_predictions:
    p = transformer_predictions[minilm_key]
    cm = confusion_matrix(p['labels'], p['predictions'], labels=[0, 1])
    minilm_diagnostic = pd.DataFrame([{
        'model': minilm_key, 'TN': int(cm[0, 0]), 'FP': int(cm[0, 1]),
        'FN': int(cm[1, 0]), 'TP': int(cm[1, 1]),
        'predicted_stress_count': int((np.asarray(p['predictions']) == 1).sum()),
        'test_count': len(p['labels'])
    }])
    minilm_diagnostic.to_csv(REVIEW_DIR / 'reviewer_minilm_collapse.csv', index=False)
    display(minilm_diagnostic)
else:
    print('Run the original MiniLM head-only experiment above to obtain this audit.')

## Validation-Only Learning-Rate Sweep

Each model/method tests three learning rates using seed 42 and up to five
epochs; validation F1 selects the setting, with ROC-AUC breaking ties.
Smaller GPUs use batch 4 with gradient accumulation 2 to preserve effective
batch 8. Completed configurations are saved incrementally.


In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer, set_seed
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import f1_score, roc_auc_score
from IPython.display import display

REVIEW_BACKBONES = ('MPNet', 'DistilBERT', 'MiniLM')
REVIEW_METHODS = ('full_fine_tuning', 'LoRA-large')
REVIEW_LRS = {
    'full_fine_tuning': (1e-5, 2e-5, 5e-5),
    'LoRA-large': (1e-4, 2e-4, 5e-4),
}
REVIEW_SEEDS = (42, 123, 2026, 7, 13, 101, 202, 404, 777, 1729)
REVIEW_MAX_EPOCHS = 5
REVIEW_EFFECTIVE_BATCH = 8
REVIEW_WEIGHT_DECAY = .01
REVIEW_WARMUP_RATIO = .10

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Runtime > Change runtime type before training.')

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
REVIEW_TRAIN_BATCH = 8 if gpu_memory_gb >= 14 else 4
REVIEW_ACCUMULATION = REVIEW_EFFECTIVE_BATCH // REVIEW_TRAIN_BATCH
assert REVIEW_TRAIN_BATCH * REVIEW_ACCUMULATION == 8

design = {
    'backbones': REVIEW_BACKBONES, 'methods': REVIEW_METHODS,
    'learning_rates': REVIEW_LRS, 'seeds': REVIEW_SEEDS,
    'epochs_max': REVIEW_MAX_EPOCHS, 'effective_batch': REVIEW_EFFECTIVE_BATCH,
    'actual_batch': REVIEW_TRAIN_BATCH, 'grad_accumulation': REVIEW_ACCUMULATION,
    'weight_decay': REVIEW_WEIGHT_DECAY, 'warmup_ratio': REVIEW_WARMUP_RATIO,
    'max_length': MAX_LENGTH,
    'train_text_hash': hashlib.sha256('\n'.join(model_train_df.text).encode()).hexdigest(),
    'validation_text_hash': hashlib.sha256('\n'.join(model_val_df.text).encode()).hexdigest(),
    'test_text_hash': hashlib.sha256('\n'.join(model_test_df.text).encode()).hexdigest(),
}
design_path = REVIEW_DIR / 'reviewer_run_design.json'
if design_path.exists():
    with open(design_path) as f:
        previous_design = json.load(f)
    if previous_design != json.loads(json.dumps(design)):
        raise RuntimeError('Existing outputs come from another design. Use a fresh '
                           'Colab runtime/output directory; do not mix experiments.')
else:
    design_path.write_text(json.dumps(design, indent=2))
print('Review GPU:', torch.cuda.get_device_name(0))
print('Matched effective batch:', REVIEW_EFFECTIVE_BATCH,
      '(per-device=', REVIEW_TRAIN_BATCH, 'accumulation=', REVIEW_ACCUMULATION, ')')

def reviewer_train(backbone, method, lr, seed, include_test=False):
    set_seed(int(seed))
    checkpoint = MODEL_CHECKPOINTS[backbone]
    tokenizer, datasets = get_tokenized_datasets(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=2,
        id2label={0: 'Non-stress', 1: 'Stress/Stress-related'},
        label2id={'Non-stress': 0, 'Stress/Stress-related': 1}
    )
    if method == 'LoRA-large':
        architecture = get_lora_model_settings(model)
        model = get_peft_model(model, LoraConfig(
            task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
            lora_dropout=.10, bias='none',
            target_modules=architecture['target_modules'],
            modules_to_save=architecture['modules_to_save']
        ))
    elif method == 'full_fine_tuning':
        for parameter in model.parameters():
            parameter.requires_grad_(True)
    else:
        raise ValueError(method)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    safe_name = f'{backbone}_{method}_lr{lr:.0e}_seed{seed}'
    output_dir = REVIEW_DIR / '_temporary_checkpoints' / safe_name
    output_dir.mkdir(parents=True, exist_ok=True)
    args = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=float(lr),
        per_device_train_batch_size=REVIEW_TRAIN_BATCH,
        gradient_accumulation_steps=REVIEW_ACCUMULATION,
        per_device_eval_batch_size=16,
        num_train_epochs=REVIEW_MAX_EPOCHS,
        weight_decay=REVIEW_WEIGHT_DECAY,
        warmup_steps=int(np.ceil(REVIEW_WARMUP_RATIO * REVIEW_MAX_EPOCHS *
            np.ceil(len(datasets['train']) / REVIEW_EFFECTIVE_BATCH))),
        eval_strategy='epoch', save_strategy='epoch', logging_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='f1',
        greater_is_better=True, save_total_limit=1,
        fp16=True, optim='adamw_torch', seed=int(seed), data_seed=int(seed),
        report_to=[], disable_tqdm=True,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=datasets['train'],
        eval_dataset=datasets['validation'], processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer, pad_to_multiple_of=8),
        compute_metrics=compute_trainer_metrics
    )
    try:
        trainer.train()
        val_output = trainer.predict(datasets['validation'])
        val_logits = np.asarray(val_output.predictions[0]
                                if isinstance(val_output.predictions, tuple)
                                else val_output.predictions)
        val_labels = np.asarray(val_output.label_ids).astype(int)
        val_probability = softmax_numpy(val_logits)[:, 1]
        val_predictions = val_logits.argmax(axis=1)
        validation_f1 = float(f1_score(val_labels, val_predictions, zero_division=0))
        validation_auc = float(roc_auc_score(val_labels, val_probability))
        epoch_records = [h for h in trainer.state.log_history
                         if 'eval_f1' in h]
        best_epoch = (float(max(epoch_records, key=lambda h: h['eval_f1'])['epoch'])
                      if epoch_records else float('nan'))
        record = {
            'backbone': backbone, 'method': method, 'lr': float(lr),
            'seed': int(seed), 'train_batch': REVIEW_TRAIN_BATCH,
            'gradient_accumulation': REVIEW_ACCUMULATION,
            'effective_batch': REVIEW_EFFECTIVE_BATCH,
            'max_epochs': REVIEW_MAX_EPOCHS, 'best_epoch': best_epoch,
            'validation_f1': validation_f1, 'validation_roc_auc': validation_auc,
            'trainable_parameters': int(trainable), 'total_parameters': int(total),
            'trainable_fraction': float(trainable / total),
        }
        prediction_bundle = {
            'validation_logits': val_logits, 'validation_labels': val_labels
        }
        if include_test:
            test_output = trainer.predict(datasets['test'])
            logits = np.asarray(test_output.predictions[0]
                                if isinstance(test_output.predictions, tuple)
                                else test_output.predictions)
            labels = np.asarray(test_output.label_ids).astype(int)
            metrics = calculate_binary_metrics(
                labels, logits.argmax(axis=1), softmax_numpy(logits)[:, 1])
            record.update({'test_' + key: float(value)
                           for key, value in metrics.items()})
            prediction_bundle.update(test_logits=logits, test_labels=labels)
        return record, prediction_bundle
    finally:
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
        shutil.rmtree(output_dir, ignore_errors=True)

def reviewer_append_record(csv_path, record):
    old = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    new = pd.concat([old, pd.DataFrame([record])], ignore_index=True)
    new.to_csv(csv_path, index=False)
    return new

In [ ]:
sweep_path = REVIEW_DIR / 'reviewer_validation_sweep.csv'
for backbone in REVIEW_BACKBONES:
    for method in REVIEW_METHODS:
        for lr in REVIEW_LRS[method]:
            prior = pd.read_csv(sweep_path) if sweep_path.exists() else pd.DataFrame()
            done = (not prior.empty and ((prior['backbone'] == backbone)
                    & (prior['method'] == method)
                    & np.isclose(prior['lr'].astype(float), lr, rtol=1e-9, atol=0)).any())
            if done:
                print('Resume sweep:', backbone, method, lr)
                continue
            print('SWEEP', backbone, method, lr, 'seed=42; validation only')
            record, _ = reviewer_train(backbone, method, lr, seed=42,
                                       include_test=False)
            reviewer_append_record(sweep_path, record)

sweep_df = pd.read_csv(sweep_path)
expected_sweep_rows = len(REVIEW_BACKBONES) * sum(len(REVIEW_LRS[m])
                                                  for m in REVIEW_METHODS)
assert len(sweep_df) == expected_sweep_rows
selected_configs = (sweep_df.sort_values(
    ['validation_f1', 'validation_roc_auc'], ascending=False)
    .drop_duplicates(['backbone', 'method'])
    .sort_values(['backbone', 'method']).reset_index(drop=True))
selected_configs.to_csv(REVIEW_DIR / 'reviewer_selected_hyperparameters.csv', index=False)
display(selected_configs[['backbone','method','lr','best_epoch',
                          'validation_f1','validation_roc_auc']])

## Ten Matched Seeds and Test Evaluation

Retrain both selected methods using the same ten seeds. Save validation
and test logits for statistical comparisons and later calibration.


In [ ]:
test_path = REVIEW_DIR / 'reviewer_matched_test_by_seed.csv'
logits_dir = REVIEW_DIR / 'seed_logits'
logits_dir.mkdir(exist_ok=True)
for backbone in REVIEW_BACKBONES:
    for method in REVIEW_METHODS:
        settings = selected_configs.loc[
            (selected_configs.backbone == backbone) &
            (selected_configs.method == method)
        ].iloc[0]
        for seed in REVIEW_SEEDS:
            key = f'{backbone}_{method}_seed{seed}'
            npz_path = logits_dir / f'{key}.npz'
            previous = pd.read_csv(test_path) if test_path.exists() else pd.DataFrame()
            completed = (not previous.empty and npz_path.exists() and
                         ((previous.backbone == backbone) &
                          (previous.method == method) &
                          (previous.seed.astype(int) == seed)).any())
            if completed:
                print('Resume final:', key)
                continue
            print('FINAL TEST:', key, 'LR:', settings.lr)
            record, arrays = reviewer_train(backbone, method, float(settings.lr),
                                            int(seed), include_test=True)
            np.savez_compressed(npz_path, **arrays)
            reviewer_append_record(test_path, record)

test_df = pd.read_csv(test_path)
expected_final_rows = len(REVIEW_BACKBONES) * len(REVIEW_METHODS) * len(REVIEW_SEEDS)
assert len(test_df) == expected_final_rows, (len(test_df), expected_final_rows)
assert not test_df.duplicated(['backbone','method','seed']).any()

metric_cols = ['test_accuracy', 'test_f1', 'test_roc_auc',
               'test_precision', 'test_recall']
summary = (test_df.groupby(['backbone', 'method'])
           .agg({column: ['mean', 'std'] for column in metric_cols}))
summary.columns = ['_'.join(item) for item in summary.columns]
summary = summary.reset_index()
summary.to_csv(REVIEW_DIR / 'reviewer_matched_summary.csv', index=False)
display(summary)

In [ ]:
from itertools import product

test_rows = []
for backbone in REVIEW_BACKBONES:
    for metric in ('test_f1', 'test_roc_auc'):
        pivot = (test_df[test_df.backbone == backbone]
                 .pivot(index='seed', columns='method', values=metric)
                 .reindex(REVIEW_SEEDS))
        assert pivot.notna().all().all()
        differences = (pivot['LoRA-large'] - pivot['full_fine_tuning']).to_numpy()
        assert len(differences) == 10
        observed = float(differences.mean())
        signs = np.asarray(list(product((-1, 1), repeat=len(differences))))
        perm_means = (signs * differences).mean(axis=1)
        exact_p = float(np.mean(np.abs(perm_means) >= abs(observed) - 1e-12))
        rng = np.random.default_rng(20261022)
        sample_indices = rng.integers(0, len(differences), size=(20000, len(differences)))
        boot_means = differences[sample_indices].mean(axis=1)
        ci_low, ci_high = np.quantile(boot_means, [.025, .975])
        test_rows.append({
            'backbone': backbone, 'metric': metric, 'n_paired_seeds': 10,
            'mean_lora_minus_full': observed, '95pct_seed_bootstrap_low': ci_low,
            '95pct_seed_bootstrap_high': ci_high,
            'unadjusted_exact_two_sided_p': exact_p,
        })

stat_df = pd.DataFrame(test_rows)
ordered = stat_df['unadjusted_exact_two_sided_p'].sort_values().index.tolist()
m = len(ordered)
adjusted = np.zeros(m)
running_max = 0.
for rank, index in enumerate(ordered):
    running_max = max(running_max,
        min(1., (m-rank)*stat_df.loc[index, 'unadjusted_exact_two_sided_p']))
    stat_df.loc[index, 'holm_adjusted_p'] = running_max
stat_df['interpretation'] = np.where(stat_df.holm_adjusted_p < .05,
    'Difference detected in this paired-seed experiment',
    'Not statistically resolved; NOT proof of equivalence')
stat_df.to_csv(REVIEW_DIR / 'reviewer_paired_statistics.csv', index=False)
display(stat_df)

## Temperature Scaling and Selective Prediction

Fit one temperature per model/seed using 423 validation examples. Report
ECE, Brier score, and coverage/accuracy at confidence thresholds 0.50–0.90
on the unchanged 715-example official test set. Empty retained subsets have
undefined accuracy and G-mean.


In [ ]:
from sklearn.metrics import brier_score_loss

calibration_rows = []
selective_rows = []
for backbone in REVIEW_BACKBONES:
    for method in REVIEW_METHODS:
        for seed in REVIEW_SEEDS:
            logits_file = logits_dir / f'{backbone}_{method}_seed{seed}.npz'
            with np.load(logits_file) as data:
                val_logits = data['validation_logits']
                val_labels = data['validation_labels']
                test_logits = data['test_logits']
                labels = data['test_labels']
            temperature, p = temperature_scaled_probabilities(
                val_logits, val_labels, test_logits)
            raw_p = softmax_numpy(test_logits)[:, 1]
            def local_ece(targets, positive_probabilities, bins=10):
                confidence = np.maximum(positive_probabilities, 1-positive_probabilities)
                correct = ((positive_probabilities >= .5).astype(int) == targets)
                edges = np.linspace(.5, 1., bins+1)
                ece = 0.
                for j in range(bins):
                    mask = ((confidence >= edges[j]) &
                            (confidence <= edges[j+1] if j == bins-1 else
                             confidence < edges[j+1]))
                    if mask.any():
                        ece += mask.mean() * abs(correct[mask].mean()-confidence[mask].mean())
                return float(ece)
            calibration_rows.append({
                'backbone': backbone, 'method': method, 'seed': seed,
                'temperature': temperature,
                'ece_before': local_ece(labels, raw_p),
                'ece_after': local_ece(labels, p),
                'brier_before': brier_score_loss(labels, raw_p),
                'brier_after': brier_score_loss(labels, p),
            })
            prediction = (p >= .5).astype(int)
            confidence = np.maximum(p, 1-p)
            for threshold in (.5, .6, .7, .8, .9):
                keep = confidence >= threshold
                if keep.any():
                    selected_y = labels[keep]
                    selected_pred = prediction[keep]
                    tn, fp, fn, tp = confusion_matrix(
                        selected_y, selected_pred, labels=[0, 1]).ravel()
                    sensitivity = tp/(tp+fn) if tp+fn else float('nan')
                    specificity = tn/(tn+fp) if tn+fp else float('nan')
                    gmean = np.sqrt(sensitivity*specificity)
                    accuracy = float((selected_y == selected_pred).mean())
                else:
                    accuracy = float('nan')
                    gmean = float('nan')
                selective_rows.append({
                    'backbone': backbone, 'method': method, 'seed': seed,
                    'threshold': threshold, 'retained_n': int(keep.sum()),
                    'test_n': len(labels), 'coverage': float(keep.mean()),
                    'retained_accuracy': accuracy, 'retained_gmean': gmean,
                })

calibration_df = pd.DataFrame(calibration_rows)
selective_df = pd.DataFrame(selective_rows)
calibration_df.to_csv(REVIEW_DIR / 'reviewer_calibration_by_seed.csv', index=False)
selective_df.to_csv(REVIEW_DIR / 'reviewer_selective_prediction_by_seed.csv', index=False)
selective_summary = (selective_df.groupby(['backbone','method','threshold'])
                     [['coverage','retained_accuracy','retained_gmean']]
                     .agg(['mean','std']))
selective_summary.columns = ['_'.join(pair) for pair in selective_summary.columns]
selective_summary = selective_summary.reset_index()
selective_summary.to_csv(REVIEW_DIR / 'reviewer_selective_prediction_summary.csv', index=False)
print('Matched-condition confidence comparison at 0.90:')
display(selective_summary[(selective_summary.backbone == 'MPNet') &
                          (selective_summary.threshold == .9)])

In [ ]:
import matplotlib.pyplot as plt
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

fig, ax = plt.subplots(figsize=(9, 5))
mpnet = selective_summary[selective_summary.backbone == 'MPNet']
for method, color in (('full_fine_tuning', 'tab:blue'),
                      ('LoRA-large', 'tab:orange')):
    subset = mpnet[mpnet.method == method].sort_values('threshold')
    ax.plot(subset.coverage_mean, subset.retained_accuracy_mean,
            '-o', label=method, color=color)
ax.set(xlabel='Mean test coverage', ylabel='Mean retained accuracy',
       title='MPNet matched training: calibrated selective prediction (10 seeds)')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(REVIEW_DIR / 'reviewer_mpnet_accuracy_coverage.png', dpi=200)
plt.show()

source_notes = [
    '# Reviewer-response manuscript notes (replace no figures by hand)',
    'The original Dreaddit benchmarks were taken from Turcan and McKeown (2019), Table 6, and were not rerun.',
    'The original and matched experiments differ in hyperparameter selection; do not mix their table rows.',
    'Training/validation/test rows are 2394/423/715. Train cleaning removed 18 repeated-text rows and 3 test-overlap rows.',
    'Matched study used effective batch 8 for both full fine-tuning and LoRA-large,',
    'five maximum epochs, best epoch/checkpoint on validation F1, and three learning rates per arm.',
    'There were ten paired seeds; exact two-sided paired sign-flip tests were Holm-corrected over six comparisons.',
    'Temperature scaling fitted on validation logits alone. Coverage/accuracy computed only on the official test.',
    'LoRA used q/v projection adapters; other target modules were not ablated.',
    'Three-seed p-values from the original manuscript should be removed or described as structurally uninformative.',
    'Do not call failure to reject a null hypothesis proof that the methods are equivalent.',
    'The compiled original PDF includes numbered citation markers. Correct Dreddit filename to Dreaddit.',
    '', '## Actual matched-condition summary',
    summary.to_string(index=False),
    '', '## Actual paired statistical comparisons',
    stat_df.to_string(index=False),
    '', '## Actual MPNet 0.90 calibrated-confidence outcomes',
    selective_summary[(selective_summary.backbone == 'MPNet') &
                      (selective_summary.threshold == .9)].to_string(index=False),
]
(REVIEW_DIR / 'reviewer_manuscript_notes.txt').write_text('\n'.join(source_notes))

archive_path = Path('/content/dreaddit_round1_reviewer_results.zip')
with ZipFile(archive_path, 'w', compression=ZIP_DEFLATED) as archive:
    for result in sorted(REVIEW_DIR.rglob('*')):
        if result.is_file():
            archive.write(result, arcname=str(result.relative_to(REVIEW_DIR)))
print('Review outputs:', REVIEW_DIR)
print('Download all reviewer tables, per-seed logits, figure and notes:')
display(FileLink(str(archive_path)))
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Open the ZIP with the link above or through the Colab Files sidebar.')